# CD8 annotation
Finalized june 8th 2026

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
# from matplotlib_venn import venn2
from scipy.stats import spearmanr
import matplotlib.patches as mpatches

from bioservices import KEGG
from gprofiler import GProfiler

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered_annotated.h5mu")

# Fix catpat (assign NIC, IC and MS annotations)

In [ ]:
#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
sdata["gex"].obs["catpat"] = pd.Categorical(np.where(sdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(sdata["gex"].obs["donor"].isin(control_list), "NIC", np.where(sdata["gex"].obs["donor"].isin(IC_list), "IC", "Error"))))


# Select CD8 T cells

In [ ]:
mu.pp.filter_obs(sdata["gex"], "final_ann", lambda x: (x.isin(["CD8 T cells"])))  
sdata.update()

In [ ]:
sdata

# Step 1: Removing doublets based on conventional QC

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 4
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)

# Plot individual values  
sc.pl.violin(
    sdata["gex"], 
    keys="n_genes_by_counts",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    sdata["gex"], 
    keys="total_counts",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    sdata["gex"], 
    keys="pct_counts_mt",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

sc.pl.violin(
    sdata["gex"], 
    keys="scrublet_score",
    groupby="Compartment",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[3]
)


#n_genes_total (i.e. number of genes per cell)
axs[0].axhline(y=200, linewidth = 1.5, color = "black", linestyle = "--")
axs[0].axhline(y=3000, linewidth = 1.5, color = "black", linestyle = "--")

#number of total reads 
axs[1].axhline(y=400, linewidth = 1.5, color = "black", linestyle = "--")
axs[1].axhline(y=4000, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[2].axhline(y=10, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[3].axhline(y=0.5, linewidth = 1.5, color = "black", linestyle = "--")

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

In [ ]:
# ## filter to remove doublets
mu.pp.filter_obs(sdata["gex"], 'n_genes_by_counts', lambda x: (x < 3000)) 
mu.pp.filter_obs(sdata["gex"], 'total_counts', lambda x: (x < 4000))
mu.pp.filter_obs(sdata["gex"], 'scrublet_score', lambda x: (x < 0.5))
sdata.update()

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("chain_pairing")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="chain_pairing",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=True,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")
plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1))

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
plt.savefig(
    "../../05_plots/11_CD8annotation/1_Violin_chain_pairing.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

# Step 2: Basic annotations

In [ ]:
df_total = sdata["gex"].obs.groupby(["Compartment", "catpat", "donor"], observed=True).size()
df_total

## CITEseq annotations

In [ ]:
# Proteins to transfer
proteins = [
    "CD45RA_adt",
    "CD4_adt",
    "CD8_adt",
    "CD62L_adt",
    "CD95_adt",
]

# Extract protein expression matrix
prot_df = pd.DataFrame(
    sdata["prot"][:, proteins].X.toarray()
    if hasattr(sdata["prot"].X, "toarray") else sdata["prot"][:, proteins].X,
    index=sdata["prot"].obs_names,
    columns=[f"prot_{p}" for p in proteins]
)

# Transfer into gex.obs
sdata["gex"].obs = sdata["gex"].obs.join(prot_df)

In [ ]:
#Also transfer the donor-dependent assignments of surface differentiation status. 
sdata["gex"].obs["Differentiation"] = sdata["prot"].obs["Differentiation"].copy()

# Step 3: Preprocess GEX

In [ ]:
# ---
# Remove genes that are only present in 1 cell
# ---
mu.pp.filter_var(sdata["gex"], 'n_cells_by_counts', lambda x: x > 1)
sdata.update()

# ---
# Allign the modalities emphasizing cells in gex.
#   Make certain that the cells in airr and prot modalities are the same cells as are present in gex.
# ---
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")
sdata.update()

# ---
# When the .var object has been filtered, then you need to renormalize
# ---
sdata["gex"].X = sdata["gex"].layers["counts"].copy()
sdata["gex"].uns.pop("log1p", None) #Removes log1P if it exists
sc.pp.normalize_total(sdata["gex"], target_sum=1e4, exclude_highly_expressed=False)   #1e4 is important. Do not change
sc.pp.log1p(sdata["gex"])

## Calculate clone_size

In [ ]:
## Calculate clone_size = number of rows per clone_id
sdata["gex"].obs["clone_size"] = sdata["gex"].obs.groupby("clone_handle", observed=True)["clone_handle"].transform("count")

#Mask "NaN" containing clones
sdata["gex"].obs["clone_size"] = np.where(sdata["gex"].obs["clone_handle"].str.contains("nan"), 0, sdata["gex"].obs["clone_size"])

## Share labels with airr
sdata["airr"].obs["clone_size"] = sdata["gex"].obs["clone_size"].copy()

## Select highly variable genes
* https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html
* https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.neighbors.html
* https://www.sc-best-practices.org/cellular_structure/integration.html

Data integration when experimental parameters vary. Ours experimental parameters are the same for each batch (apart from Tcsf being fresh cells, and Tpb being rested). We prefer to work with unintegrated data wherever possible to preserve differences between samples. The link describes the re-normalization of of genes, selection of hvg, ect. we need to do the same. 

### Batch aware highly variable genes using default functions

* Source: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html
  
_For flavor='seurat_v3_paper', genes are first sorted by the number of batches a gene is a HVG, with ties broken by the median (across batches) rank_

* Source: https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.htm

**batch_key**
str | None (default: None)
If specified, highly-variable genes are selected within each batch separately and merged. This simple process avoids the selection of batch-specific genes and acts as a lightweight batch correction method. For **all flavors**, except seurat_v3, **genes are first sorted by how many batches they are a HVG.** For dispersion-based flavors ties are broken by normalized dispersion. For flavor = 'seurat_v3_paper', ties are broken by the median (across batches) rank based on within-batch normalized variance.

**For flavor="seruat" (default)**: Effector genes of known function are a big part of the HVGs with high mean expression and dispersion values fx, GZMK and PRF are both above 5 in mean expression value. Many low dispersion genes are unknown of B / macrophage genes. I will prioritize a method that includes many known T cell genes

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
sdata["gex"].var = sdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
sdata["gex"].obs["batch_handle"] = sdata["gex"].obs["batch"].astype(str) + "_" + sdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(sdata["gex"], 
                            n_top_genes=6500, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a too dense umap.
                            flavor='seurat',  
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended. 
                            span=1)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD8_HVGs_export_before_TCR_filter.csv")

## Remove TCR genes from the list of highly variable genes

In [ ]:
#Bring gene names from index into a column
sdata["gex"].var = sdata["gex"].var.reset_index()

# Optionally rename the new column
sdata["gex"].var = sdata["gex"].var.rename(columns={"index": "genes"})

In [ ]:
# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = sdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = sdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
sdata["gex"].var["highly_variable"] = np.where(sdata["gex"].var["genes"].isin(masked_genes), False, sdata["gex"].var["highly_variable"])

# Return the genes to index
sdata["gex"].var = sdata["gex"].var.set_index(["genes"], drop=True)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD8_HVGs_export_after_TCR_filter.csv")

## PCR, neighours and **over**clustering
### Calculate neighbors
https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pp.neighbors.html
- n_neighbors int (default: 15)

    The size of local neighborhood (in terms of number of neighboring data points) used for manifold approximation. **Larger values result in more global views of the manifold**, while smaller values result in more local data being preserved. In general values should be in the range 2 to 100. If knn is True, number of nearest neighbors to be searched. If knn is False, a Gaussian kernel width is set to the distance of the n_neighbors neighbor.

#### General advice: Larger datasets benefit from larger n_pcs values! Source: https://satijalab.org/seurat/archive/v3.0/mca
- Here they use n_pcs = 75 for 242.000 cells, and they are generally very lax about where to set the cutoff on the elbow plot.

Advice: Try working with larger n_pcs. You can decrease the n_neighbors to 10, if you want to reproduce Seurats behavior

In [ ]:
# Run PCA
sc.pp.pca(sdata["gex"], svd_solver="arpack", use_highly_variable=True, n_comps=60)

In [ ]:
# Visualize pca
sc.pl.pca_scatter(sdata["gex"], color=["total_counts", "GZMK", "GZMB", "GZMA"])

In [ ]:
sc.pl.pca_variance_ratio(sdata["gex"], n_pcs=60, log=True) #Log tranformeD. 

In [ ]:
# Set n_pcs based on a visual impression
sc.pp.neighbors(sdata["gex"], n_pcs=40)

In [ ]:
sc.tl.leiden(sdata["gex"], key_added="leiden_4", resolution=4.0)

## Calculate umap

In [ ]:
sc.tl.umap(sdata["gex"], spread=1)

### Reporting on T-T doublets

In [ ]:
confusion_matrix = pd.crosstab(
    sdata["gex"].obs["leiden_4"],
    sdata["gex"].obs["chain_pairing"],
    rownames=['leiden_4'],
    colnames=['chain_pairing']
)


# Calculate row-wise percentages for the "two full chains" column
confusion_matrix["percent_two full chains"] = (
    confusion_matrix["two full chains"] / confusion_matrix.sum(axis=1)
) * 100

confusion_matrix


## Doublet QC plots

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_4")["n_genes_by_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="n_genes_by_counts",
        groupby="leiden_4",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("n_genes_by_counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_4")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="leiden_4",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()


#Cluster 28 do have more counts - but this cluster has a very low fraction of two-chain TCRs. So I think it's okay. 

In [ ]:
sc.pl.umap(sdata["gex"], color=["leiden_4"], groups=["14","45"], size = 10, legend_fontweight= 'bold', frameon= False , vmin=0, legend_fontoutline=2, show=True)

## Return cell numbers

In [ ]:
## How many cells
sdata["gex"].obs.groupby(["final_ann"]).size()

In [ ]:
## How many cells
sdata["gex"].obs.groupby(["chain_pairing"]).size()

#### Remove the doublet cluster

In [ ]:
### Exclude doublets when only CD8 T cells are studied
# mu.pp.filter_obs(sdata["gex"], "chain_pairing", lambda x: (x != "two full chains")) 
mu.pp.filter_obs(sdata["gex"], "leiden_4", lambda x: (x != "45")) 

#Remove unused cateorgies
sdata["gex"].obs["leiden_4"] = sdata["gex"].obs["leiden_4"].cat.remove_unused_categories()
sdata.update()

## Generate the final umap/clusters

In [ ]:
#Removes columns of the var table. This is to rebuild the HVG table. 
sdata["gex"].var = sdata["gex"].var[[]]

## What is a batch?
## We want a donor aware + batch aware intergration (donor + batch)
sdata["gex"].obs["batch_handle"] = sdata["gex"].obs["batch"].astype(str) + "_" + sdata["gex"].obs["donor"].astype(str)

## Default highly variable genes
sc.pp.highly_variable_genes(sdata["gex"], 
                            n_top_genes=6500, #!! 5000 genes creates a spread out umap. 7500 HVGs creates a more dense umap.
                            flavor='seurat',  #This mimics select intergration features
                            batch_key="batch_handle",   #If a batch was "batch" - then the CITEseq multiplexed instances would immediate mark donor-specific expression of AC004448.2 and OOEP fx. this is not intended. 
                            span=1)

Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD8_HVGs_export_before_TCR_filter.csv")

#### Remove TCR genes from the list of highly variable genes

In [ ]:
#Bring gene names from index into a column
sdata["gex"].var = sdata["gex"].var.reset_index()

# Optionally rename the new column
sdata["gex"].var = sdata["gex"].var.rename(columns={"index": "genes"})

# Rename the highly_varaible_intersection cells according to an ifelse statement
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRBV', "TRBD", "TRBJ"] 

pattern = r"^(" + "|".join(tcr_gene_prefixs) + ")"
mask = sdata["gex"].var["genes"].str.contains(pattern, regex=True)
masked_genes = sdata["gex"].var[mask]["genes"].tolist() #Works - it's giving me the exact genes I want except the D-region genes (although there is only 2? I will ask Kilian about it. Lea says there is something like that with mice, that htey don't express it)

#Modify the initial list of highly variable genes to exclude TCR genes
sdata["gex"].var["highly_variable"] = np.where(sdata["gex"].var["genes"].isin(masked_genes), False, sdata["gex"].var["highly_variable"])

# Return the genes to index
sdata["gex"].var = sdata["gex"].var.set_index(["genes"], drop=True)

In [ ]:
Export_HVG = sdata["gex"].var[sdata["gex"].var["highly_variable"]==True].sort_values(by="highly_variable_nbatches", ascending = True)
print(len(Export_HVG))
Export_HVG.to_csv("..\\..\\06_csv_outputs\\HVG\\CD8_HVGs_export_after_TCR_filter.csv")

#### Redo PCA, umap and leiden

In [ ]:
# Run PCA
sc.pp.pca(sdata["gex"], svd_solver="arpack", use_highly_variable=True, n_comps=60)
sc.pl.pca_variance_ratio(sdata["gex"], n_pcs=60, log=True) #Log tranformeD. 


In [ ]:
# Set n_pcs based on a visual impression
sc.pp.neighbors(sdata["gex"], n_pcs=40)

#umap
sc.tl.umap(sdata["gex"], spread=1)

In [ ]:
# sc.tl.leiden(sdata["gex"], key_added="leiden_1", resolution=1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.1", resolution=1.1) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.2", resolution=1.2) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.3", resolution=1.3) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.4", resolution=1.4) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.5", resolution=1.5)
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.6", resolution=1.6) 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.7", resolution=1.7) 
sc.tl.leiden(sdata["gex"], key_added="leiden_1.8", resolution=1.8)      #Works well - the outlier clusters are clearly marked in both CSF and PB without too many clusters to account for. 
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.9", resolution=1.9)      
# sc.tl.leiden(sdata["gex"], key_added="leiden_2", resolution=2.0)    
# sc.tl.leiden(sdata["gex"], key_added="leiden_1.8", resolution=2.1)  
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.2", resolution=2.2)
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.3", resolution=2.3)
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.4", resolution=2.4)
# sc.tl.leiden(sdata["gex"], key_added="leiden_2.5", resolution=2.5)

In [ ]:
sc.pl.umap(sdata["gex"], color=[#"leiden_1.3", "leiden_1.4", "leiden_1.5",  "leiden_1.6",  "leiden_1.7","leiden_1.9", "leiden_2", "leiden_1.8"
    "leiden_1.8"], size = 5, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data")


# Return umaps

In [ ]:
cluster_color_dict = {
    # --- Pastel colors / TN ---
    "0":  "#dbe9f6",  # very light blue
    "1":  "#c7e9c0",  # pastel green
    "2":  "#e2d9f3",  # light lavender
    "11": "#fdd0a2",  # pastel peach

    # --- TCMs ---
    "7":  "#74c476",  # green
    "13": "#31a354",  # dark green
    "21": "#3BFD16",  # neon

    # --- TEM ---
    "4":  "#fdae6b",  # light orange
    "8":  "#fd8d3c",  # orange

    # --- TEMRA ---
    "9":  "#fb6a4a",  # soft red
    "10": "#8B0000",  # dark red

    # --- TE ---
    "6":  "#C00000",  # red
    "12": "#843c39",  # brown red

    # --- Activated ---
    "17": "#f768a1",  # pink
    "19": "#c51b8a",  # 
    "20": "#FF1B76",  # 
    "22": "#d9d9d9",  # 

    # --- Gamma delta ---
    "15": "#737373",  # Dark grey

    # --- CSF blue / purple-blue gradient ---
    "3":  "#6baed6",  # light blue
    "5":  "#3182bd",  # medium blue
    "14": "#9e9ac8",  # light purple
    "16": "#756bb1",  # purple
    "18": "#1f4fbf",  # dark blue

}

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["leiden_1.8"], size = 5, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, 
                      palette=cluster_color_dict, legend_loc="on data", cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/11_CD8annotation/2_leiden_umap.pdf", bbox_inches="tight", dpi=400)
plt.show()


# Step 4: Cluster annotations
- Show marker genes
- Remake CITEseq annotations
- Annotate broad phenotypes (Tn, Tem ect)

In [ ]:
sdata["gex"].obs["leiden_1.8"].value_counts()

In [ ]:
### CITEseq
marker_genes = {
         "CITEseq": ["prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         "Naive & memory markers": ["TCF7", "LEF1", "SELL", "CCR7", "IL7R", "KLF2", "MYC", "BACH2", "LTB"],
          "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
         "Co-stimulation" : ["CD27", "CD28", "ITGB2", "ITGAL", "TNFRSF4"],
         "Effector functions": ["CCL5", "NKG7", "GZMB", "PRF1", "GNLY", "GZMA", "GZMH", "GZMK",], 
         "Differentiation": ["CX3CR1", "LAG3", "TIGIT",  "IKZF2", "EOMES", "ZEB2", "TOX",  "PRDM1", "PDCD1", "CD74", "HLA-DRB1"], 
         "Innate-receptors" : ["KLRD1", "KLRK1", "KLRC1","KLRC2", "NCR3"], 
          "Resident memory": ["ITGAE", "ZNF683", "XCL1"],
         "Activation": ["CD69", "NR4A1"],
         "Signaling": ["NFKB2","NFKBIA", "RELA", "RELB", "NFATC1", "NFATC2", "TGFB1", "FOS", "FOSB", "JUN", "MAF", "MAFG", "ATF2",  "BATF3", "ATF4", "EIF1", "BTG1", "MRPS6", "SH3BGRL3", "BAX"], 
        "Integrins" : ["ITGA4", "ITGB1", "ITGAE", "ITGB7"],
        "Homing" : ["CXCR3", "CXCR4", "CXCR6", "CCR4", "CCR5"], 
        "Cell cycle" : ["MKI67", "CDKN1B"],
    "General markers": ["MALAT1", "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8","TRDV1"],
    "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
}

# # Define a custom order:
cluster_order = [
    "2", "0", "1",  "11",
    "7", "13", 
    "4", "8",
   "10",  "9", 
    "6", "12",
    "17", "20", "19","22",
    "15", "21",
    "18", "14", "3", "5", "16", 
]

# #Implement order
sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
    cluster_order, ordered=True
)

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/5_CD8_bubblechart.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
#All naive and memory markers
# sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.8"].isin(["0", "1", "2", "6", "15" ]), "Tn (0, 1, 2, 6, 15)", "non-Tn"))
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["5"], reference="rest", key_added="upX")
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)


## Compare CITEseq information with transcriptomic information

In [ ]:
obs_df = sdata["gex"].obs.copy()

# Exclude clusters from CSF
obs_df = obs_df[~obs_df["leiden_1.8"].isin(["3", "5", "14", "16","18"])]

# Get grouped counts per differentiation assignment
sum_df = obs_df.groupby(["Differentiation", "leiden_1.8"], observed=True).size().reset_index(name="count")

# Add sums per leiden cluster
sum_df["total"] = sum_df.groupby(["leiden_1.8"],observed=True)["count"].transform("sum")

# Calculate percentages
sum_df["freq"] = 100 * sum_df["count"] / sum_df["total"]

# Pivot to wide format: rows = leiden_1.8, columns = Differentiation
plot_df = (
    sum_df
    .pivot(index="leiden_1.8", columns="Differentiation", values="freq")
    .fillna(0)
)

# Sort leiden clusters numerically if needed
plot_df = plot_df.sort_index()

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4),
    width=0.75
)

ax.set_xlabel("leiden_1.8")
ax.set_ylabel("Frequency (%)")
ax.set_title("Differentiation based on CD45RA, CD62L, and CD95 surface expression")

ax.legend(
    title="Subset",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/6_CITEseq_gated_phenotypes.pdf", bbox_inches="tight",  dpi=400)
plt.show()

## Show protein and gene expression side-by-side

In [ ]:
sc.pl.umap(sdata["gex"], color=["prot_CD45RA_adt", "PTPRC"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/11_CD8annotation/9_CD8_PTPRC_CD45RA_protein.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
sc.pl.umap(sdata["gex"], color=["prot_CD62L_adt", "SELL"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/11_CD8annotation/10_CD8_SELL_CD62L_A_protein.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
sc.pl.umap(sdata["gex"], color=["prot_CD95_adt", "FAS"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/11_CD8annotation/11_CD8_FAS_CD95_protein.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
sc.pl.umap(sdata["gex"], color=["prot_CD8_adt", "CD8A"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/11_CD8annotation/12_CD8_CD8A_CD8_protein.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
# fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# states = ["Tn", "Tscm", "Tcm", "Tem", "Temra"]

# for ax, state in zip(axes, states):
#     sc.pl.umap(
#         sdata["gex"],
#         color="Differentiation",
#         groups=[state],        # single group only
#         size=10,
#         frameon=False,
#         legend_loc=None,
#         show=False,
#         ax=ax,
#         title=state,
#     )

# plt.tight_layout()

# fig.savefig(
#     "../../05_plots/11_CD8annotation/13_CD8_umaps_highlighted_states.pdf",
#     bbox_inches="tight"
# )

## Flow-like CITE-seq plots

In [ ]:
## How many CD8 T cells do I expect? 
len(sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"])

In [ ]:
#Transfer L3_ann_overview to prot
sdata["prot"].obs["leiden_1.8"] =  sdata["gex"].obs["leiden_1.8"].copy()

#Work only with pb cells
sdata_pb = sdata.copy()

mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB"))

In [ ]:
#It's more correct to not extract it again. We need to use the prot_ values in GEX_obs
df = sdata_pb["gex"].obs[["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt", "leiden_1.8", "Compartment"]].copy()

# # -----------------------------------
# # FEATURES TO PLOT
# # -----------------------------------
# features = ["CD45RA_adt", "CD62L_adt", "CD95_adt"]

# # -----------------------------------
# # EXTRACT PER-CELL ADT VALUES
# # -----------------------------------
# # Get column indices in var
# idx = [sdata_pb["prot"].var_names.get_loc(f) for f in features]

# # Handle sparse or dense .X
# X = sdata_pb["prot"].X
# if not isinstance(X, np.ndarray):
#     X = X.toarray()

# # Build dataframe (cells × 2 ADTs)
# df = pd.DataFrame(
#     X[:, idx],
#     columns=features,
#     index=sdata_pb["prot"].obs_names
# )

# # Keep only non-NaN data points
# df = df.dropna()

# # -----------------------------------
# # ADD L3 ANNOTATION
# # -----------------------------------
# df["leiden_1.8"] = sdata_pb["prot"].obs.loc[
#     df.index, "leiden_1.8"
# ]

# # Keep only non-NaN data points
# df = df.dropna()

In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
CD45RA_gate = 14
CD62L_gate = 13
xlim = (-3, 51)
ylim = (-3, 41)

# -----------------------------------
# Example plots to set thresholds
# -----------------------------------
g = sb.jointplot(
    data=df,
    x="prot_CD45RA_adt",
    y="prot_CD62L_adt",
    kind="kde",
    fill=True,
    bw_adjust=0.50,
    levels=10,
    thresh=0.05,
    clip=((-5, 50), (-5, 40)),
    height=5
)

# Move legend outside. 
# sb.move_legend(
#     g.ax_joint,
#     "upper left",
#     bbox_to_anchor=(1.4, 1),
#     frameon=False,
#     title="L3 annotation"
# )

ax = g.ax_joint

# -----------------------------------
# GATING LINES and despine
# -----------------------------------
ax.axvline(CD45RA_gate, color="black", linestyle="--")
ax.axhline(CD62L_gate, color="black", linestyle="--")

sb.despine(ax=ax)

# -----------------------------------
# Limits
# -----------------------------------
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# -----------------------------------
# Save the figure
# -----------------------------------
plt.tight_layout()
g.savefig("../../05_plots/11_CD8annotation/46_CD45RA_CD62L_global_gate.pdf", 
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------------
# PARAMETERS (Normalized ADT SPACE)
# -----------------------------------
CD95_gate = 9
CD62L_gate = 13
xlim = (-10, 51)
ylim = (-3, 41)

# -----------------------------------
# Example plots to set thresholds
# -----------------------------------
g = sb.jointplot(
    data=df,
    x="prot_CD95_adt",
    y="prot_CD62L_adt",
    kind="kde",
    # hue="L3_ann_overview",
    fill=True,
    bw_adjust=0.50,
    levels=10,
    thresh=0.05,
    clip=((-10, 50), (-5, 40)),
    height=5
)

# Move legend outside. 
# sb.move_legend(
#     g.ax_joint,
#     "upper left",
#     bbox_to_anchor=(1.3, 1),
#     frameon=False,
#     title="L3 annotation"
# )

ax = g.ax_joint

# -----------------------------------
# GATING LINES and despine
# -----------------------------------
ax.axvline(CD95_gate, color="black", linestyle="--")
ax.axhline(CD62L_gate, color="black", linestyle="--")

sb.despine(ax=ax)

# -----------------------------------
# Limits
# -----------------------------------
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)

# -----------------------------------
# Save the figure
# -----------------------------------
plt.tight_layout()
g.savefig("../../05_plots/11_CD8annotation/47_CD95_CD62L_global_gate.pdf", 
    dpi=400,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------------
# Collective gates
# -----------------------------------
CD95_gate = 9
CD62L_gate = 13
CD45RA_gate = 14

# -----------------------------------
# Add qudrant and CD95 annotation
# -----------------------------------
df["Quadrant"] = pd.Categorical(np.where(
    (df["prot_CD45RA_adt"] >= CD45RA_gate) & (df["prot_CD62L_adt"]  >= CD62L_gate), "Tn-like", np.where(
        (df["prot_CD45RA_adt"] < CD45RA_gate) & (df["prot_CD62L_adt"]  >= CD62L_gate), "Tcm: CD45RA-CD62L+", np.where(
            (df["prot_CD45RA_adt"] < CD45RA_gate) & (df["prot_CD62L_adt"]  < CD62L_gate), "Tem: CD45RA-CD62L-", np.where(
                (df["prot_CD45RA_adt"] >= CD45RA_gate) & (df["prot_CD62L_adt"] < CD62L_gate), "Temra: CD45RA+CD62L-", "error")))))

df["CD95_ann"] = pd.Categorical(np.where(
    df["prot_CD95_adt"] >= CD95_gate, "CD95+", "CD95-"))

df["Quadrant_final"] = pd.Categorical(np.where(
    (df["Quadrant"] == "Tn-like") & (df["CD95_ann"] == "CD95+"), "Tscm: CD45RA+CD62L+CD95+", np.where(
        (df["Quadrant"] == "Tn-like") & (df["CD95_ann"] == "CD95-"), "Tn: CD45RA+CD62L+CD95-", df["Quadrant"])))

df["Quadrant_final"].value_counts()

### Plots for each individual cluster

In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
x_gate = 14
y_gate = 13

xlim = (-5, 61)
ylim = (-5, 41)

# -----------------------------------
# LOOP OVER L3 CATEGORIES
# -----------------------------------
for l3 in df["leiden_1.8"].dropna().unique():

    df_sub = df[df["leiden_1.8"] == l3]

    if len(df_sub) < 50:
        print(f"Skipping {l3} (n={len(df_sub)})")
        continue

    x = df_sub["prot_CD45RA_adt"]
    y = df_sub["prot_CD62L_adt"]

    # -----------------------------------
    # JOINT KDE PLOT (new figure per L3)
    # -----------------------------------
    g = sb.jointplot(
        data=df_sub,
        x="prot_CD45RA_adt",
        y="prot_CD62L_adt",
        kind="kde",
        fill=True,
        bw_adjust=0.50,
        levels=10,
        thresh=0.05,
        clip=((-5, 60), (-5, 40)),
        height=5
    )

    ax = g.ax_joint

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # -----------------------------------
    # GATING LINES
    # -----------------------------------
    ax.axvline(x_gate, color="black", linestyle="--")
    ax.axhline(y_gate, color="black", linestyle="--")

    sb.despine(ax=ax)

    # -----------------------------------
    # QUADRANT PERCENTAGES (PER L3)
    # -----------------------------------
    total = len(df_sub)

    q1 = (x >= x_gate) & (y >= y_gate)
    q2 = (x < x_gate) & (y >= y_gate)
    q3 = (x < x_gate) & (y < y_gate)
    q4 = (x >= x_gate) & (y < y_gate)

    pcts = [
        100 * q1.sum() / total,
        100 * q2.sum() / total,
        100 * q3.sum() / total,
        100 * q4.sum() / total,
    ]

    # -----------------------------------
    # QUADRANT ANNOTATION
    # -----------------------------------
    dx = 0.05 * (xlim[1] - xlim[0])
    dy = 0.10 * (ylim[1] - ylim[0])

    ax.text(x_gate + dx, y_gate + dy, f"{pcts[0]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate + dy, f"{pcts[1]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate - 2*dy, f"{pcts[2]:.1f}%", fontsize=14)
    ax.text(x_gate + dx, y_gate - 2*dy, f"{pcts[3]:.1f}%", fontsize=14)

    # -----------------------------------
    # TITLE + EXPORT
    # -----------------------------------
    ax.set_title(f"{l3} (n={total:,})")

    plt.tight_layout()
    plt.savefig(
        f"../../05_plots/08_AfterAntwerp/64_CD8_CD45RA_CD62L_{l3}.pdf",
        dpi=400,
        bbox_inches="tight"
    )
    plt.show()


### Correspondance
How well my GEX annotations recapitulate CITEseq annotations?

In [ ]:
df_sub[df_sub["Quadrant_final"] == "error"]

In [ ]:
###### Skip cluster 3,6,7, and 19, which are cells in CSF
df_sub = df[~df["leiden_1.8"].isin(["3", "4", "18", "19","20"])]

# Crosstab (rows = Quadrant, columns = L3 annotation)
ct = pd.crosstab(df_sub["leiden_1.8"], df_sub["Quadrant_final"])

# Sort rows by total number of cells["3", "4", "18", "19","20"]
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=True).index]

# Plot horizontal stacked bar chart
ax = ct.plot(
    kind="barh",       # horizontal bars
    stacked=True,
    figsize=(8, 5),
    colormap="tab20"   # nice color palette
)

plt.ylabel("Leiden clusters")  # now vertical axis
plt.xlabel("Number of cells")          # horizontal axis
plt.title("")
plt.legend(title="CITEseq annotations", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(
    "../../05_plots/11_CD8annotation/14_CD8_CITEseq_reannotation_leiden_1.8.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# # Define a custom order:
Cluster_order = [
    "2", "0", "1",  "11",
    "7", "13", 
    "4", "8",
   "10",  "9", 
    "6", "12",
    "17", "20", "19","22",
    "15", "21",
    "18", "14", "3", "5", "16", 
]


# #Implement order
df_sub["leiden_1.8"] = df_sub["leiden_1.8"].cat.reorder_categories(
    Cluster_order, ordered=True)

# Crosstab (rows = L3 annotation, columns = Quadrant)
ct = pd.crosstab(df_sub["Quadrant_final"], df_sub["leiden_1.8"])

quadrant_order = [
    "Tn: CD45RA+CD62L+CD95-",
    "Tscm: CD45RA+CD62L+CD95+",
    "Tcm: CD45RA-CD62L+",
    "Tem: CD45RA-CD62L-",
    "Temra: CD45RA+CD62L-"
]

# Reindex columns according to desired order
ct = ct.reindex(index=quadrant_order)

# Optionally normalize per row for proportions
ct_norm = ct.div(ct.sum(axis=0), axis=1) * 100  # percentages

# Plot heatmap
plt.figure(figsize=(14, 3))
sb.heatmap(
    ct_norm,
    annot=True,        # show numbers
    fmt=".1f",         # one decimal
    cmap="Reds",    # color map
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "% of cells per L3 annotation"}
)

plt.ylabel("CITEseq annotations")
plt.xlabel("Leiden 2.1")
plt.title("Phenotypic distribution across L3 clusters")
plt.tight_layout()
plt.savefig(
     "../../05_plots/11_CD8annotation/15_CD8_CITEseq_reannotation_correspondance.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()

## Return these annotations to the AnnData objecct

In [ ]:
### Add the quadrant_final labels back to sdata
# -----------------------------------
# Collective gates
# -----------------------------------
CD95_gate = 9
CD62L_gate = 13
CD45RA_gate = 14

# -----------------------------------
# Add qudrant and CD95 annotation
# -----------------------------------
sdata["gex"].obs["Quadrant"] = pd.Categorical(np.where(
    (sdata["gex"].obs["prot_CD45RA_adt"] >= CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"]  >= CD62L_gate), "Tn-like", np.where(
        (sdata["gex"].obs["prot_CD45RA_adt"] < CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"]  >= CD62L_gate), "Tcm: CD45RA-CD62L+", np.where(
            (sdata["gex"].obs["prot_CD45RA_adt"] < CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"]  < CD62L_gate), "Tem: CD45RA-CD62L-", np.where(
                (sdata["gex"].obs["prot_CD45RA_adt"] >= CD45RA_gate) & (sdata["gex"].obs["prot_CD62L_adt"] < CD62L_gate), "Temra: CD45RA+CD62L-", "error")))))

sdata["gex"].obs["CD95_ann"] = pd.Categorical(np.where(
    sdata["gex"].obs["prot_CD95_adt"] >= CD95_gate, "CD95+", "CD95-"))

sdata["gex"].obs["Quadrant_final"] = pd.Categorical(np.where(
    (sdata["gex"].obs["Quadrant"] == "Tn-like") & (sdata["gex"].obs["CD95_ann"] == "CD95+"), "Tscm: CD45RA+CD62L+CD95+", np.where(
        (sdata["gex"].obs["Quadrant"] == "Tn-like") & (sdata["gex"].obs["CD95_ann"] == "CD95-"), "Tn: CD45RA+CD62L+CD95-", sdata["gex"].obs["Quadrant"])))

sdata["gex"].obs["Quadrant_final"].value_counts()

In [ ]:
obs_df = sdata["gex"].obs.copy()

# Exclude clusters from CSF
obs_df = obs_df[~obs_df["leiden_1.8"].isin(["3", "4", "18", "19","20"])]
obs_df = obs_df[~obs_df["Compartment"].isin(["CSF"])]

# Get grouped counts per differentiation assignment
sum_df = obs_df.groupby(["Quadrant_final", "leiden_1.8"], observed=True).size().reset_index(name="count")

# Add sums per leiden cluster
sum_df["total"] = sum_df.groupby(["leiden_1.8"],observed=True)["count"].transform("sum")

# Calculate percentages
sum_df["freq"] = 100 * sum_df["count"] / sum_df["total"]

# Pivot to wide format: rows = leiden_1.8, columns = Differentiation
plot_df = (
    sum_df
    .pivot(index="leiden_1.8", columns="Quadrant_final", values="freq")
    .fillna(0)
)

# Sort leiden clusters numerically if needed
plot_df = plot_df.sort_index()

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 4),
    width=0.75
)

ax.set_xlabel("leiden_1.8")
ax.set_ylabel("Frequency (%)")
ax.set_title("Differentiation based on CD45RA, CD62L, and CD95 surface expression")

ax.legend(
    title="Subset",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/16_CITEseq_gated_phenotypes_UPDATED.pdf", bbox_inches="tight",  dpi=400)
plt.show()

## Implement broader annotations

In [ ]:
cluster_dict = {
    "0": "TN",
    "1": "TN",
    "2": "TN",
    "11": "TN",

    "7": "TCM",
    "13": "TCM",

    "18": "TCSF",
    "14": "TCSF",
    "3": "TCSF",
    "5": "TCSF",
    "16": "TCSF",
    
    "4": "TEM",
    "8": "TEM",
    "12": "TE",
    "6": "TE",
    "9": "TEMRA",
    "10": "TEMRA",
    
    "17": "TA",
    "19": "TA",
    "20": "TA",
    "22": "TA",

    "21" : "NK-like", 
    "15" : "GDT cells"
}

# Add cell type column based on annotation
sdata["gex"].obs['L3_ann_overview'] = [cluster_dict[sample] for sample in sdata["gex"].obs['leiden_1.8']]

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="L3_ann_overview"
)

plt.show()

## Implement exact anntations

In [ ]:
# cluster_dict = {
   
#     "1":  r"T_N (STAT1^hi IFN^hi)",
#     "2":  r"T_N (STAT1^hi IFN^int)",
#     "0":  r"T_N (STAT1^lo IFN^lo)",
#     "5":  r"T_N (STAT1^int IFN^lo)",

    
#     "10": r"T_CM (LTB^hi IFN^lo)",
#     "17": r"T_CM (LTB^hi IFN^hi)",
#     "13": r"T_CM (CCR4^hi IFN^int)",

#     # --- TCSF ---- 
#     "18": r"T_CSF (TCF7^hi ITGA4^int)",
#     "20": r"T_CSF (CD27^hi ITGA4^hi)",
#     "3":  r"T_CSF (GZMK^hi HLA-DR^hi)",
#     "19": r"T_CSF (CXCR6^hi XCL1^hi)",
#     "4":  r"T_CSF (IL7R^hi CD69^hi)",

#     # --- TEM ---- 
#     "15": r"T_EM (LTB^hi IFN^lo)",
#     "9":  r"T_EM (GZMK^hi IFN^lo)",
#     "11": r"T_EM (GZMK^hi IFN^hi)",

#      # --- TE and TEMRA ---- 
#     "7":  r"T_EMRA (EOMES^hi IFN^hi)",
#     "8":  r"T_EMRA (EOMES^int IFN^lo)",
#     "12": r"T_E (CX3CR1^hi GNLY^lo)",
#     "6":  r"T_E (CX3CR1^hi GNLY^hi)",


#     # --- TA - activated ---
#     "23": r"T_Act (CD69^hi JUN/FOS^hi)",
#     "25": r"T_Act (Ribo^low MALAT1^hi)",
#     "22": r"T_Act (TCF7^hi NFκB^hi)",
#     "14": r"T_Act (TCR7^hi BAX^hi)",
#     "21": r"T_Act (CD69^hi TGFB1^hi)",

#      # --- Other ---
#     "16": r"γδ T cells",
#     "24": r"T_NK (NCR3^hi)",
# }

cluster_dict = {

    # --- TN ---
    "2":  r"TN (BACH2hi IFNlo)",
    "0":  r"TN (BACH2int IFNlo)",
    "1":  r"TN (BACH2int IFNhi)",
    "11": r"TN (MRPS6hi)",

    # --- TCM ---
    "7":  r"TCM (LTBhi IFNlo)",
    "13": r"TCM (LTBhi IFNhi)",

    # --- TEM ---
    "4":  r"TEM (GZMKhi IFNlo)",
    "8":  r"TEM (GZMKhi IFNhi)",

    # --- TEMRA ---
    "10": r"TEMRA (EOMESlo IFNlo)",
    "9":  r"TEMRA (EOMEShi IFNhi)",

    # --- TE ---
    "12": r"TE (GZMBhi GNLYlo)",
    "6":  r"TE (GZMBhi GNLYhi)",

    # --- Other ---
    "15": r"γδ T cells",
    "21": r"TNK-like (NCR3hi)",

    # --- TAct ---
    "17": r"TAct (TCR7hi NFKB2hi)",
    "20": r"TAct (CD69hi TGFB1hi)",
    "19": r"TAct (CD69hi JUN/FOShi)",
    "22": r"TAct (Ribolow MALAT1hi)",

    # --- TCSF ---
    "18": r"TCSF (TCF7hi LTBhi)",
    "14": r"TCSF (GZMKhi IL7Rhi)",
    "3":  r"TCSF (GZMKhi HLA-DRB1hi)",
    "5":  r"TCSF (GZMKhi Ribohi)",
    "16": r"TCSF (CXCR6hi XCL1hi)"
}

# Add cell type column based on annotation
sdata["gex"].obs['L3_ann'] = [cluster_dict[sample] for sample in sdata["gex"].obs['leiden_1.8']]


In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="L3_ann"
)

plt.show()

# Step 5: Reporting

## Compartments

In [ ]:
#color dictionary
Compartment_map = {
    "PB": "#C06A6A",   # red
    "CSF":  "#4B8AC1",   # blue
}

sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="Compartment",
    palette=Compartment_map,
    size=10,
    legend_fontweight="bold",
    legend_fontsize=6,
    frameon=False,
    show=False,
    legend_loc="on data",
    title=""
)

# Save exactly like your example
plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/4_Compartment_CSF_PB_umap.pdf",
    bbox_inches="tight",
    dpi=400
)
plt.show()

## Gini index
The different index metrics are described here: https://scikit.bio/docs/dev/generated/skbio.diversity.alpha.html

The function to calculate these metrics is described here: https://scirpy.scverse.org/en/latest/generated/scirpy.tl.alpha_diversity.html

**Gini index** portrays dominance defined as: Dominance [5] (or concentration) measures the degree that one or a few most abundant species (taxa) represent the great majority of a community. It can be considered as a measure of community unevenness.

In [ ]:
sdata.update()

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

sdata.update()

In [ ]:
sdata_gini = sdata.copy()
sdata_gini["airr"].obs["clone_handle"] = sdata_gini["gex"].obs["clone_handle"].copy()
sdata_gini["airr"].obs["leiden_1.8"] = sdata_gini["gex"].obs["leiden_1.8"].copy()
sdata_gini["airr"].obs["donor"] = sdata_gini["gex"].obs["donor"].copy()
sdata_gini["airr"].obs["catpat"] = sdata_gini["gex"].obs["catpat"].copy()

#Make grouping variables
sdata_gini["airr"].obs["donor_leiden_1.8"] = sdata_gini["airr"].obs["donor"].astype(str) + "-" + sdata_gini["airr"].obs["leiden_1.8"].astype(str)

#filter
mu.pp.filter_obs(
    sdata_gini["airr"],
    "clone_id",
    lambda x: ~pd.isna(x)
)

In [ ]:
sdata_gini["airr"].obs[["clone_handle"]]

## Diversity per leiden

In [ ]:
## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_gini["airr"], groupby="leiden_1.8", target_col='clone_handle', metric='gini_index', inplace=True, key_added="gini_per_cluster", airr_mod='airr')

In [ ]:
# Drop rows with NaN gini index
df_gini = sdata_gini["airr"].obs[["leiden_1.8", "gini_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["gini_per_cluster"])
df_gini["gini_per_cluster"] = pd.to_numeric(df_gini["gini_per_cluster"])

# df_gini.sort_values("gini_per_cluster")
order = df_gini.sort_values("gini_per_cluster",ascending=False)["leiden_1.8"].to_list()

In [ ]:
df_gini["leiden_1.8"] = df_gini["leiden_1.8"].astype(str)

plt.figure(figsize=(8,5))

sb.barplot(
    data=df_gini,
    x='leiden_1.8',
    y='gini_per_cluster',
    order=order,
    color='lightgrey'
)

plt.xlabel("leiden_1.8")
plt.ylabel("Gini index")
plt.title("Gini per Leiden cluster")
# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/17_Gini_per_cluster.pdf", bbox_inches="tight",  dpi=400)
plt.show()

# Diversity per general cluster

In [ ]:
### Transfer the L3_ann_broad to airr
sdata_gini["airr"].obs["L3_ann_overview"] = sdata_gini["gex"].obs["L3_ann_overview"].copy()

In [ ]:
## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_gini["airr"], groupby="L3_ann_overview", target_col='clone_handle', metric='gini_index', inplace=True, key_added="gini_per_cluster", airr_mod='airr')

In [ ]:
# Drop rows with NaN gini index
df_gini = sdata_gini["airr"].obs[["L3_ann_overview", "gini_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["gini_per_cluster"])
df_gini["gini_per_cluster"] = pd.to_numeric(df_gini["gini_per_cluster"])

# df_gini.sort_values("gini_per_cluster")
order = df_gini.sort_values("gini_per_cluster",ascending=False)["L3_ann_overview"].to_list()

In [ ]:
df_gini["L3_ann_overview"] = df_gini["L3_ann_overview"].astype(str)

plt.figure(figsize=(8,5))

sb.barplot(
    data=df_gini,
    x='L3_ann_overview',
    y='gini_per_cluster',
    order=order,
    color='lightgrey'
)

plt.xlabel("L3_ann_overview")
plt.ylabel("Gini index")
plt.title("Gini per Leiden cluster")
# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/17_Gini_per_cluster_broadly.pdf", bbox_inches="tight",  dpi=400)
plt.show()

## Work signatures
List of libraries that can be imported: https://maayanlab.cloud/Enrichr/#libraries

This script was made using Claude!

### Apoptosis (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04210')  # Apoptosis for humans
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Apoptosis (KEGG)')

### Ribosomes (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa03008')  # Ribosomes
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Ribosomes (KEGG)')

### Ribosome biogenesis (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0042254"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Ribosome biogenesis (GOBP)')

### Translation elongation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0006414"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Translational elongation (GOBP)')

### Translation initiation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0006413"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Translational initiation (GOBP)')

### Execution phase of apoptosis (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0097194"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='Execution phase of apoptosis (GOBP)')

### Base exicision repair (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa03410')  # Base excision repair.
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Base excision repair (KEGG)')

### Oxidative phosphorylation (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa00190')  # Oxphos
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Oxidative phosphorylation (KEGG)')

### Glycolysis/ gluconeogenesis (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa00010')  # Base excision repair.
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Glycolysis and gluconeogenesis (KEGG)')

### Pentose phosphate pathway (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa00030')  # Pentose phosphate pathway
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Pentose phosphate pathawy (KEGG)')

### Proteasome (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa03050')  # PRoteasome
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Proteasome (KEGG)')

### T cell differentiation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0030217"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='T cell differentiation (GOBP)')

### T cell activation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0042110"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='T cell activation (GOBP)')

### Regulation of actin cytoskeleton (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04810')  # Regulation of actin cytoskeleton
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Regulation of actin cytoskeleton (KEGG)')

### T-cell prolifferation (GOBP)

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0042098"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='T cell prolifferation (GOBP)')

### Cell Cycle-Associated Genes (KEGG)

In [ ]:
# Retriving a select pathway
k = KEGG()

# Get pathway information
pathway = k.get('hsa04110')  # Cytokine-cytokine receptor interaction 
parsed = k.parse(pathway)

# Extract genes
genes = []
if 'GENE' in parsed:
    for gene_id, gene_info in parsed['GENE'].items():
        genes.append({
            'id': gene_id,
            'name': gene_info.split(';')[0]
        })

print(f"Found {len(genes)} genes")
gene_list = [d["name"] for d in genes]

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      gene_list,  
                      score_name='Cell cycle - Homo sapiens (KEGG)')

### Cytokine-cytokine receptor interaction (KEGG)

In [ ]:
# # Retriving a select pathway
# k = KEGG()

# # Get pathway information
# pathway = k.get('hsa04060')  # Cytokine-cytokine receptor interaction 
# parsed = k.parse(pathway)

# # Extract genes
# genes = []
# if 'GENE' in parsed:
#     for gene_id, gene_info in parsed['GENE'].items():
#         genes.append({
#             'id': gene_id,
#             'name': gene_info.split(';')[0]
#         })

# print(f"Found {len(genes)} genes")
# gene_list = [d["name"] for d in genes]

# # Cacluate a score
# sc.tl.score_genes(sdata["gex"], 
#                       gene_list,  
#                       score_name='Cytokine-cytokine receptor interaction (KEGG)')

### T CELL_RECEPTOR_SIGNALING_PATHWAY 

In [ ]:
## I use GProfiler for this. 
gp = GProfiler(return_dataframe=True)

go_term = "GO:0050852"

res = gp.convert(
    organism="hsapiens",
    target_namespace="ENSG",
    query=go_term
)

genes = res["name"].unique().tolist()
print("I have this many genes:", len(genes))

# Cacluate a score
sc.tl.score_genes(sdata["gex"], 
                      genes,  
                      score_name='TCR signaling (GOBP)')

### Cytotoxicity and interferon scores

In [ ]:
## Cytotoxicity score
Toxic_score = ["GZMB", "GZMH","GZMA", "PRF1", "GNLY", "NKG7"]

sc.tl.score_genes(sdata["gex"], 
                      Toxic_score,  
                      score_name='Cytotoxic\nscore')


In [ ]:
## CANONICAL_NF_KAPPAB_SIGNAL_TRANSDUCTION (GOBP)
NFKB_sig = [
    "TANK","MIR365A","HUWE1","ADAM8","OPTN","NAMPT","TRIM13","FLOT1","TNIP1","TLR6",
    "TFG","TRIM22","NOD1","TAB1","TRIM38","UBD","LAMTOR5","PRDX4","OLFM4","SIVA1",
    "RBCK1","NFAT5","TRAF3IP2","PLK2","ZMYND11","NEK6","USP20","MALT1","EDAR","LILRB4",
    "PIM2","MID2","FAF1","IRAK3","AKAP13","TMEM106A","CHRNA7","TIRAP","CHUK","CARD16",
    "CCR7","CANT1","IRAK1BP1","CREBBP","SIRPA","PARP1","MIB2","TMC8","PPM1N","TICAM1",
    "CTH","CX3CR1","DAB2IP","CYLD","DDX1","DHX15","DHX36","NLRP6","ZNF675","ECM1",
    "ECT2","EDA","LPAR1","NLRC3","EP300","ESR1","F2R","F2RL1","TMED4","UNC5CL",
    "FKBP1A","CARD8","TRIM32","MAPKBP1","ERC1","TAB2","FLNA","FLOT2","FBXW11","UFL1",
    "ZDHHC17","SIRT1","BRD4","NUP62","IFIT5","DEFB124","ABL1","TAB3","FBXO7","GAPDH",
    "PYDC1","ANKRD17","GJA1","ATP2C1","SLCO3A1","ANGPT1","NKIRAS2","NKIRAS1","LURAP1L","TRIM59",
    "CCDC22","PYCARD","TBK1","GSTP1","CARD10","HDAC1","ANXA4","HMOX1","PRMT2","BIRC2",
    "BIRC3","XIAP","HSPB1","HTR2B","CLEC4D","JMJD8","IRGM","SLC35B2","TICAM2","IKBKB",
    "IL1A","IL1B","IL1R1","IL1RAP","FASLG","ILK","INS","INSR","IRAK1","IRAK2",
    "IRF3","RHOA","SUMO4","GBP7","RHOC","LGALS1","LGALS9","LIMS1","RHOH","MYO18A",
    "LTA","LTB","LTBR","LTF","MIR138-1","MIR140","MIR146A","MIR15A","MIR15B","MIR16-1",
    "MIR195","MIR199A1","MIR21","MIR27A","MIR30C2","ARRB2","MAS1","MAP3K3","CD200","MYD88",
    "NFKB1","NFKB2","NFKBIA","NFKBIB","NFKBIL1","ROR1","TIFAB","C18orf32","PRKN","GOLT1B",
    "IRAK4","SHISA5","TLR7","TLR8","HACD3","CXXC5","PIAS4","PDPK1","PER1","PLCG2",
    "TLR9","LURAP1","TREM2","TERF2IP","ZFAND6","ZDHHC13","NDFIP2","UGT1A1","CC2D1A","LIME1",
    "PPM1A","PPM1B","RNF31","PPP2CB","SBNO1","TRIM62","PPP5C","PIDD1","TNFRSF19","INAVA",
    "PRKCB","PRKCE","PRKD1","MAP2K5","PRL","TRIM39","TMEM9B","ZC3HAV1","SPHK2","PSMA6",
    "C15orf39","OTUD7B","SLC44A2","PELI2","PELI1","MIR497","PLEKHG5","MAVS","IFT80","MIER1",
    "CACTIN","REL","RELA","RELB","BCL3","EDA2R","RORA","S100A4","S100A12","S100A13",
    "S100B","CCL19","CCL21","CX3CL1","SECTM1","PBLD","SLC39A8","NOD2","CARD9","AZI2",
    "BLVRA","CLEC7A","LRRC19","PINK1","GPR89A","SLC20A1","DDRGK1","SPI1","STAT1","STAT3",
    "BST2","MAP3K7","BTK","TFRC","TGFB1","TGFBR3","TLE1","TLR2","TLR3","TLR4",
    "TSPAN6","TMSB4X","TNF","TNFAIP3","TNFRSF1A","NR2C2","TRAF1","TRAF2","TRAF3","TRAF5",
    "TRAF6","UBE2I","UBE2N","UBE2V1","VEGFA","WNT5A","TRIM25","CARD14","TNIP2","MUL1",
    "NLRX1","SNIP1","TNIP3","WLS","ZC3H12A","ALPK1","NDFIP1","APOL3","TRIM8","SHARPIN",
    "CAPN3","CASP1","ITCH","CASP8","CARD19","WDR83","CASP10","TMEM101","CARD11","CUL1",
    "TRIM52","AJUBA","IKBKG","TRIM5","CAV1","TNFSF11","TRADD","RIPK1","TNFSF14","TNFSF10",
    "RIPK2","FADD","RIOK3","TNFRSF11A","TNFRSF10B","CFLAR","SQSTM1","TAX1BP1","CPNE1","BCL10",
    "BTRC","DNAJA3","USP10","NLRP12","DDX21","MTDH","VAPA","TIFA","KLF4","ADIPOQ",
    "CLEC6A","CD36","LITAF","NR1D1","CD40","CD40LG","TRAF4","IKBKE","CD74","RASSF2",
    "TNFSF15","NR1H4","RBX1"
]

sc.tl.score_genes(sdata["gex"], 
                      NFKB_sig,  
                      score_name='Cannonical NFkB signal transduction (GOBP)')

IFN_ab = [
    "TANK", "PSMD14", "ADAR", "IFITM3", "IFITM2", "ADRM1", "CDC37", "USP18",
    "TREX1", "TRIM6", "CR2", "DCST1", "TRIM65", "TTLL12", "YTHDF3", "SAMHD1",
    "SIN3A", "GIGYF2", "LSM14A", "SETD2", "TBK1", "CNOT7", "UBE2K", "IFNE",
    "STING1", "IFI27", "IFIT1", "IFNA1", "IFNA2", "IFNA4", "IFNA5", "IFNA6",
    "IFNA7", "IFNA8", "IFNA10", "IFNA14", "IFNA16", "IFNA17", "IFNA21",
    "IFNAR1", "IFNAR2", "IFNB1", "IFNW1", "IRAK1", "IRF3", "IRF7", "JAK1",
    "USP27X", "SMIM30", "MIR21", "MMP12", "MX1", "MYD88", "OAS1", "OAS2",
    "OAS3", "YTHDF2", "RBM47", "SHFL", "METTL3", "PSMA1", "PSMA2", "IFNK",
    "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7", "PSMB1", "PSMB2", "PSMB3",
    "PSMB4", "GPR108", "PSMB5", "PSMB6", "PSMB7", "PSMC1", "PSMC2", "PSMC3",
    "PSMC4", "PSMC5", "PSMC6", "PSMD1", "PSMD2", "PSMD3", "PSMD4", "PSMD7",
    "PSMD8", "PSMD11", "PSMD12", "PSMD13", "PSME1", "PSME2", "MAVS", "USP29",
    "PTPN1", "PTPN2", "PTPN6", "PTPN11", "CACTIN", "IFIH1", "AZI2", "SHMT2",
    "SMPD1", "SP100", "STAT1", "STAT2", "TRAF3", "TYK2", "WNT5A", "MUL1",
    "SEM1", "ZBP1", "TRIM56", "NLRC5", "IFITM1", "OASL", "FADD", "CH25H",
    "TRIM41", "RNF185", "EIF4E2", "ISG15", "IKBKE", "TBKBP1", "HDAC4", "PSMD6"
]

sc.tl.score_genes(sdata["gex"], 
                      IFN_ab,  
                      score_name='Response to type I IFN (GOBP)')



IFN_gamma = [
    "NR1H3", "ACTR3", "ACTR2", "CALCOCO2", "IFITM3", "CDC42EP2", "SYNCRIP",
    "UBD", "IFITM2", "ADAMTS13", "CDC37", "VPS26B", "GPR146", "GBP4", "GBP5",
    "OTOP1", "SIRPA", "IL23R", "CYP27B1", "DAPK1", "DAPK3", "GBP6",
    "SLC30A8", "EDN1", "AIF1", "PDE12", "EPRS1", "FCAR", "FLNB", "RPL13A",
    "CDC42EP4", "STXBP4", "GAPDH", "GBP1", "GBP2", "GBP3", "GCH1", "GSN",
    "SEC61A1", "HCK", "HLA-DPA1", "HPX", "RAB7B", "RAB43", "IRF8", "IRGM",
    "IFNG", "IFNGR1", "IFNGR2", "FASLG", "IL12B", "IL12RB1", "AQP4", "IRF1",
    "JAK1", "JAK2", "KIF5B", "ARG1", "GBP7", "LGALS9", "LCN10", "MEFV",
    "CIITA", "CD200", "MRC1", "ABCC1", "CITED1", "ASS1", "MYO1C", "ATF3",
    "NOS2", "NUB1", "PIM1", "PARP14", "PPARG", "MED1", "SHFL", "RAB20",
    "PSMA1", "PSMA2", "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7",
    "PSMB1", "PSMB2", "PSMB3", "PSMB4", "PSMB8", "PSMB9", "PSMB10",
    "PTPN2", "CXCL16", "RAF1", "RPS6KB1", "CCL2", "CCL3", "CCL5",
    "SLC26A6", "SLC11A1", "SLC22A5", "SNCA", "SP100", "TRIM21",
    "STAT1", "STXBP1", "STXBP2", "STXBP3", "BST2", "CRIPTO", "TLR2",
    "TLR4", "ACTG1", "TNF", "TP53", "TXK", "TYK2", "ACOD1", "NR1H2",
    "VIM", "WAS", "WNT5A", "ZYX", "CALM1", "CAMK2A", "CASP1", "PARP9",
    "NLRC5", "IFITM1", "VAMP4", "KYNU", "CLDN1", "ZNF697", "DNAJA3",
    "VAMP3", "STX8", "CD40", "CD47", "CD58", "CD74", "CDC42"
]

sc.tl.score_genes(sdata["gex"], 
                      IFN_gamma,  
                      score_name='Response to type II IFN (GOBP)')

## Nonoverlapping
# Remove duplicates before computing set differences
IFN_ab_unique = list(set(IFN_ab))
IFN_Gamma_unique = list(set(IFN_gamma))

# Genes unique to each list
IFN_ab_only = sorted(list(set(IFN_ab_unique) - set(IFN_Gamma_unique)))
IFN_Gamma_only = sorted(list(set(IFN_Gamma_unique) - set(IFN_ab_unique)))

sc.tl.score_genes(sdata["gex"], 
                      IFN_ab_only,  
                      score_name='Exclusive to type I interferon signaling')
sc.tl.score_genes(sdata["gex"], 
                      IFN_Gamma_only,  
                      score_name='Exclusive to type II interferon signaling')


### Plot signatures

In [ ]:
pathway_list = sdata["gex"].obs.columns[(sdata["gex"].obs.columns.str.contains("KEGG")) | sdata["gex"].obs.columns.str.contains("GOBP") | sdata["gex"].obs.columns.str.contains("Exclusive")].tolist()

sc.pl.matrixplot(
    sdata["gex"],
    var_names=pathway_list,   # works for obs keys too
    groupby="leiden_1.8",
    use_raw=False,
    return_fig=False,
    show=True,
    swap_axes=True,
    cmap="RdBu_r",
    standard_scale="var",   # z-score per score (row-wise)
)

plt.savefig("../../05_plots/11_CD8annotation/18_CD8_leiden_pathways.pdf", bbox_inches="tight",  dpi=400)

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
sc.pl.umap(sdata["gex"], color=pathway_list, size = 10, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=True, legend_loc="on data", cmap="RdBu_r", return_fig=True)
plt.savefig("../../05_plots/11_CD8annotation/19_CD8_umap_pathways.pdf", bbox_inches="tight",  dpi=400)

## Polished bubble plots

In [ ]:
### NAIVE
marker_genes = {
         # "CITEseq": [ "prot_CD8_adt", "prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         "Naive": [ "TCF7", "LEF1", "SELL", "CCR7", "CD27", "CD28", "BACH2", "STAT1", "IFI6", "ISG20" ,"OAS1", "MX1","ISG15", "PABPC1", "LDHB", "EEF1B2", "SNHG29", "ITGA6"],
          # "Specific for 12 & 16": ["LTB", "CLDND1", "GPR171", "SESN3"],
         "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
         # "Differentiation": ["NKG7", "KLRG1", "CX3CR1", "LAG3", "TIGIT",  "IKZF2", "EOMES", "TOX",  "PRDM1", "PDCD1"], 
         # "Innate-receptors" : ["KLRD1", "KLRK1", "KLRC1","KLRC2"], 
         # "Effector molecules": [ "CCL5", "GZMK", "GZMB", "GZMH", "PRF1", "GNLY", "GZMA", "TNF" , "IFNG", "IL2"], 
          # "Resident memory": ["S1PR1", "KLF2", "ITGAE", "ZNF683"],
         # "Activation": ["CD69", "NR4A1"],
         # "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
         #  "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        # "Integrins" : ["ITGA4", "ITGB1", "ITGB2", "ITGAE", "ITGAL"], 
        # "Homing" : ["CXCR3", "CXCR4", "CXCR6", "CCR4"], 
    # "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
    # "Cell survival and growth" : ["FOXO1", "NOTCH1", "MKI67", "EGFR","BCL2L11", "BCL6", "BCL3", "BCL2"], 
# "General markers": ["CD3E", "CD8A", "IL7R", "LCK", "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8", "TRDV1"],
    # "Other" : ["PDE3B", "MALAT1"]
}


dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/20_CD8_bubblechart_naive_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### CITEseq
marker_genes = {
         "CITEseq": [ "prot_CD8_adt", "prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         # "Naive": [ "TCF7", "LEF1", "SELL", "CCR7", "CD27", "CD28", "BACH2", "ISG15"],
          # "Specific for 12 & 16": ["LTB", "CLDND1", "GPR171", "SESN3"],
         # "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
         # "Differentiation": ["NKG7", "KLRG1", "CX3CR1", "LAG3", "TIGIT",  "IKZF2", "EOMES", "TOX",  "PRDM1", "PDCD1"], 
         # "Innate-receptors" : ["KLRD1", "KLRK1", "KLRC1","KLRC2"], 
         # "Effector molecules": [ "CCL5", "GZMK", "GZMB", "GZMH", "PRF1", "GNLY", "GZMA", "TNF" , "IFNG", "IL2"], 
          # "Resident memory": ["S1PR1", "KLF2", "ITGAE", "ZNF683"],
         # "Activation": ["CD69", "NR4A1"],
         # "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
         #  "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        # "Integrins" : ["ITGA4", "ITGB1", "ITGB2", "ITGAE", "ITGAL"], 
        # "Homing" : ["CXCR3", "CXCR4", "CXCR6", "CCR4"], 
    # "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
    # "Cell survival and growth" : ["FOXO1", "NOTCH1", "MKI67", "EGFR","BCL2L11", "BCL6", "BCL3", "BCL2"], 
# "General markers": ["CD3E", "CD8A", "IL7R", "LCK", "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8", "TRDV1"],
    # "Other" : ["PDE3B", "MALAT1"]
}


# #Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/21_CD8_bubblechart_CITEseq_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### TCM
marker_genes = {
         # "CITEseq": [ "prot_CD8_adt", "prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         "TCM": [ "TCF7", "LEF1", "SELL", "CCR7", "CD27", "CD28", "BACH2", "ISG15", "OAS1", "MX1", "ITGA4", 
                  "S100A11", "ALOX5AP", "PABPC1", "PLP2", "GPR171", "LTB", "ITGB1", "CREM", "SOCS1"
],
}

# #Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/22_CD8_bubblechart_TCM_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### Specific to Tcm
marker_genes = {
         # "CITEseq": [ "prot_CD8_adt", "prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         # "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
         # "Naive": [ "TCF7", "SELL", "CCR7", "CD27"],
         "Tcm-related": ["TCF7","CCR7", "SELL","PABPC1", "IL7R", "LTB", "BACH2","GPR171", "SESN3", "CLDND1", "TNFSF8", "ITGB1", "S100A11", "CCL5"],
         "C24 specific" : ["AOAH","CD7", "IKZF2", "CXCR3", "NCR3", "ZNF683","KLRB1", "NCAM1", "FCER1G", "TNFSF14", "TNFSF10", "TNFRSF4", "TNFSF4", "KIR2DL1", "KIR2DL3", "KIR2DL4", "KIR3DL1", "KIR3DL2", "KIR3DL3"]
         # "Other": ["NKG7", "KLRG1", "CX3CR1", "CCL5", "GZMB", "GZMK"], 
         # "Innate-receptors" : ["KLRD1", "KLRK1", "KLRC1","KLRC2"], 
         # "Effector molecules": [ "CCL5", "GZMK", "GZMB", "GZMH", "PRF1", "GNLY", "GZMA", "TNF" , "IFNG", "IL2"], 
          # "Resident memory": ["S1PR1", "KLF2", "ITGAE", "ZNF683"],
         # "Activation": ["CD69", "NR4A1"],
         # "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
         #  "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        # "Integrins" : ["ITGA4", "ITGB1", "ITGB2", "ITGAE", "ITGAL"], 
        # "Homing" : ["CXCR3", "CXCR4", "CXCR6", "CCR4"], 
    # "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
    # "Cell survival and growth" : ["FOXO1", "NOTCH1", "MKI67", "EGFR","BCL2L11", "BCL6", "BCL3", "BCL2"], 
# "General markers": ["CD3E", "CD8A", "IL7R", "LCK", "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8", "TRDV1"],
    # "Other" : ["PDE3B", "MALAT1"]
}

#Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/23_CD8_bubblechart_Tcm_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### Specific to CSF
marker_genes = {
         # "CITEseq": [ "prot_CD8_adt", "prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         # "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
         # "Naive": [ "TCF7", "SELL", "CCR7", "CD27"],
         # "Tcm DEGs": ["PABPC1", "IL7R", "LTB", "GPR171", "SESN3", "CLDND1", "TNFSF8", "ITGB1", "S100A11", "CCL5"],
         # "Other": ["NKG7", "KLRG1", "CX3CR1", "CCL5", "GZMB", "GZMK"], 
         # "Innate-receptors" : ["KLRD1", "KLRK1", "KLRC1","KLRC2"], 
         "Memory" : ["TCF7", "IL7R", "SELL", "CCR7", "KLF2"], 
         "Effector molecules": ["IL7R","GZMB", "PRF1", "GNLY", "NKG7", "CCL5", "GZMA", "GZMK", "GZMH", "CST7", "EOMES"], 
         "Resident memory": ["S1PR1", "KLF2", "ITGAE", "ZNF683", "XCL1"],
         "Activation": ["CD69", "NR4A1", "HLA-DRA", "HLA-DRB1"],
         "Strong DEGs": ["TXNIP", "S100A4", "CTSW", "HCST", "LGALS1","CXCR6", "CD7", "HOPX"],
          "Co-stimulation" : ["CD27", "CD28", "ITGB2", "ITGAL"],
         # "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
         #  "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        # "Integrins" : ["ITGA4", "ITGB1", "ITGB2", "ITGAE", "ITGAL"], 
        "Homing" : ["ITGA4", "ITGB1", "CXCR3", "CXCR4", "CXCR6", "CCR4", "CCR5", "GPR183"], 
    # "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
    # "Cell survival and growth" : ["FOXO1", "NOTCH1", "MKI67", "EGFR","BCL2L11", "BCL6", "BCL3", "BCL2"], 
# "General markers": ["CD3E", "CD8A", "IL7R", "LCK", "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8", "TRDV1"],
    # "Other" : ["PDE3B", "MALAT1"]
}


#Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/24_CD8_bubblechart_CSF_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### Specific to TM-GZMK
marker_genes = {
         # "CITEseq": [ "prot_CD8_adt", "prot_CD45RA_adt", "prot_CD62L_adt","prot_CD95_adt"],
         # "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5", "MT-CYB"],
         # "Naive": [ "TCF7", "SELL", "CCR7", "CD27"],
         # "Tcm DEGs": ["PABPC1", "IL7R", "LTB", "GPR171", "SESN3", "CLDND1", "TNFSF8", "ITGB1", "S100A11", "CCL5"],
         # "Other": ["NKG7", "KLRG1", "CX3CR1", "CCL5", "GZMB", "GZMK"], 
         # "Innate-receptors" : ["KLRD1", "KLRK1", "KLRC1","KLRC2"], 
         "Memory" : ["TCF7", "SELL", "CCR7", "IL7R", "LTB"], 
         "Actin" : ["S100A6", "S100A11"], 
         "Differentiation": ["CX3CR1", "KLRG1"],
         "Effector molecules" : ["GZMB", "GZMA", "GZMK", "PRF1", "CCL5"],
         # "Chemokines" : ["CCL4", "CCL5", "XCL1"], 
         # "HLA-related" : ["CD74", "HLA-DRB1", "HLA-DPB1"],
         # "Resident memory": ["S1PR1", "KLF2", "ITGAE", "ZNF683", "XCL1"],
         # "Activation": ["CD69", "NR4A1", "HLA-DPB1"],
         "Strong DEGs": ["PPP1R14B", "CLDND1", "CD44", "PDE4B"],
         "Co-stimulation" : ["CD27", "CD28", "ITGB2", "ITGAL"],
         # "NFkB signaling": ["NFKB1", "NFKB2","NFKBIA", "RELA", "RELB"], 
         #  "NFAT/AP-1 signaling": ["NFATC1", "NFATC2",  "FOS", "FOSB", "JUN", "JUNB", "JUND", "TGFB1", "MAF", "MAFG", "ATF2", "BATF", "BATF2", "BATF3"],
        # "Integrins" : ["ITGA4", "ITGB1", "ITGB2", "ITGAE", "ITGAL"], 
        "Homing" : ["ITGA4", "ITGB1", "CXCR3", "CXCR4", "CXCR6", "CCR4", "CCR5", "GPR183"], 
    "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
    # "Cell survival and growth" : ["FOXO1", "NOTCH1", "MKI67", "EGFR","BCL2L11", "BCL6", "BCL3", "BCL2"], 
# "General markers": ["CD3E", "CD8A", "IL7R", "LCK", "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8", "TRDV1"],
    # "Other" : ["PDE3B", "MALAT1"]
}

#Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/25_CD8_bubblechart_TEM_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### Specific to activated and ribolo
marker_genes = {
        "DEGs" : ["CCR7", "SARAF", "CYTIP", "MT-ND2", "MALAT1", "MRPS6", "NELL2", "BAX", "LEPROTL1"],
         "Ribosomal & mito": ["RPL30", "RPL28", "RPL11", "MT-ATP8", "MT-ND5", "MT-CYB"],
         "Memory" : ["TCF7", "IL7R", "SELL", "CCR7"], 
         "Activation": ["CD69", "NR4A1", "HLA-DPB1"],
         "Signaling" : ["NFATC1", "NFATC2", "NFKB1", "NFKB2", "NFKBIA", "NFKBIZ", "RELA", "RELB", "JUNB", "FOS", "TNFAIP3", "ATF2", "BATF",  "BATF3"],
         "Apoptosis" : ["STK17A"], 
         "Strong DEGs": ["CXCR6","ACTB", "ACTG1", "S100A11", "S100A6", "METRNL", "TGFB1", "PIK3R1", "PIK3IP1","MALAT1", "MBNL1"], 
         "Heatshock DEGs" : ["HSPA1B", "HSPB1", "HSPA1A", "HSP90AB1", "HSPA8", "HSPA5"],
         "Microtubles" : ["TUBA1C", "TUBB4B"],
         "Cell cycle and proliferation" : ["MKI67", "CCNL1", "CDKN1A", "CDKN1B", "CDKN1C", "CREBBP", "CDK12", "CDK17", "ANAPC7", "ANAPC10"],
         "Inhibition of cell cycle" : ["CDKN2AIP"],
    "Interferon responses": ["IRF1", "ISG15", "OAS1", "MX1", "STAT1"],
"General markers": ["CD3E", "CD8A", "CD8B", "HCST","LCK" , "TRAC", "TRBC1", "TRBC2", "TRGV3", "TRGV8", "TRDV1"],

}


#Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/27_CD8_bubblechart_Activation_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
t_cell_tfs = [
    "TCF7",
    "LEF1",
    "FOXO1",
    "BACH2",
    "KLF2",
    "FOXP1",
    "TOX",
    "TBX21",
    "RUNX3",
    "BATF",
    "ZEB2",
    "ID2",
    "STAT4",
    "PRDM1",
    "NR4A1",
    "NR4A2",
    "NR4A3",
    "MAF",
    "IKZF2",
    "BCL6",
    "AHR",
    "BCL11B",
    "TCF3",
    "RUNX1",
    "EOMES",
    "BCL3"
]

dp = sc.pl.dotplot(
    sdata["gex"],
    t_cell_tfs,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/27_CD8_TFs_focus.pdf",
    bbox_inches="tight",
    dpi=400
)


In [ ]:
### Specific to activated and ribolo
marker_genes = {
    "CITEseq" : ["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"], 
    "TN" : ["SELL", "CCR7", "BACH2"],
    "IFN" : ["ISG15", "MX1", "STAT1"],
    "TCM" : ["TCF7", "LTB", "CCR7"], 
    "TEM" : ["CD27", "GZMA", "GZMK"], 
    "TE": ["GNLY", "GZMB", "NKG7", "PRF1", "GZMH", "EOMES", "TBX21", "CX3CR1", "IKZF2"],
    "NK-like" : ["KIR3DL2", "KIR3DL1", "TYROBP", "KLRD1", "KLRC2"],
    "GD" : ["TRDV1"], 
    "CSF-TE" : ["GZMK", "CD74", "HLA-DRA", "HLA-DRB1", "IL7R", "ITGA4", "ITGB1"],
    "CSF-TR" : ["CD69", "ZNF683", "XCL1", "CXCR6"],
    "TA": ["NFKBIA", "RELA", "RELB", "BATF3", "TGFB1", "TNFAIP3", "JUNB", "FOS"],
    "Ribosomal & mito": ["RPL28", "RPL11", "MT-ATP8", "MT-ND5"],
    "General" : ["CD3E", "CD8A", "CD8B", "HCST","LCK"]
    
}

# Define a custom order:
# Cluster_order = ["0", "1", "2", "6", "15",  #Tn
#                 "24", "11", "7",   #Tcm
#                  "8","5",         #Tem
#                  "13", "14",       #Temra
#                  "10", "12", "9",      #Effector cells
#                  "18",                      #TRDV1+ cells
#                  "17","23", "20","21", "25", 
#                  "3", "4", "16", "19", "22" ]    #Tcsf]    #Activated

# #Implement order
# sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.reorder_categories(
#     Cluster_order, ordered=True
# )

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="leiden_1.8",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/27_CD8_general.pdf",
    bbox_inches="tight",
    dpi=400
)


## Step 3: Global differential gene expression analysis

In [ ]:
sc.tl.rank_genes_groups(sdata["gex"], 'leiden_1.8', method='wilcoxon', key_added="leiden_1.8_DEG")
Export_DEG = sc.get.rank_genes_groups_df(sdata["gex"], group = None, key = "leiden_1.8_DEG")
Export_DEG.to_csv("..\\..\\06_csv_outputs\\DEG\\CD8s_Leiden2_ann_DEG_n50_DEGs.csv")


In [ ]:
sc.pl.rank_genes_groups(sdata["gex"], key="leiden_1.8_DEG")

### Format the table for differentially expressed genes. 

In [ ]:
DEGs_export = (
    Export_DEG[
        (Export_DEG["logfoldchanges"] > 1) &
        (Export_DEG["pvals_adj"] < 0.01)
    ]
    .sort_values("scores", ascending=False)
)

# gene_lists = (
#     DEGs_export
#         .groupby("group")
#         .head(50)
#         .groupby("group")["names"]
#         .apply(list)
# )

# gene_lists_df = pd.DataFrame({
#     group: pd.Series(genes)
#     for group, genes in gene_lists.items()
# })

# gene_lists_df.to_excel("../../06_csv_outputs/DEG/CD8s_Leiden2_all_sig_degs_padj_below_0.05_lfc_above_1.xlsx")


# Number of clusters
n_clusters = 26

# Initialize empty list to hold DataFrames for each cluster
dfs = []

for cluster in range(n_clusters):
    cluster_str = str(cluster)
    
    # Extract top 30 genes for each dataset
    top_export = DEGs_export[DEGs_export["group"] == cluster_str].dropna().sort_values("scores", ascending=False)["names"].tolist()
    top_filtered = DEGs_export[DEGs_export["group"] == cluster_str].dropna().sort_values("scores", ascending=False)["names"].tolist()
    
    # Pad shorter list to match max length
    max_len = max(len(top_export), len(top_filtered))
    top_export += [""] * (max_len - len(top_export))
    top_filtered += [""] * (max_len - len(top_filtered))
    
    # Create DataFrame for this cluster
    df_cluster = pd.DataFrame({
        f"Cluster {cluster}": top_export,
        f"Cluster {cluster}": top_filtered
    })
    
    dfs.append(df_cluster)

# Concatenate all clusters column-wise
df_all = pd.concat(dfs, axis=1)

# Export to Excel
df_all.to_excel("../../06_csv_outputs/DEG/CD8s_Leiden2.1_all_sig_degs_padj_below_0.01_lfc_above_1.xlsx", index=False)

print("✅ Excel file 'Leiden_clusters_top_genes.xlsx' created with all clusters appended column-wise!")


In [ ]:
fig_deg = sc.pl.rank_genes_groups_dotplot(
    sdata["gex"], groupby="leiden_1.8", standard_scale="var", n_genes=10, key="leiden_1.8_DEG", cmap="RdBu_r", show=False, return_fig=True, dendrogram=False)

fig_deg.savefig("../../05_plots/11_CD8annotation/28_CD8_leiden2_DEG_bubblechart.pdf", bbox_inches="tight",  dpi=400)
plt.show()


## Build figures using umaps showcasing the most prominently expressed markers for annotation

In [ ]:
#Main figure material
fig_markers = sc.pl.umap(sdata["gex"], color=["TCF7","CCR7", "ITGA4", "LTB", 
                                             "GZMK", "CXCR3", "CD27", "TIGIT",
                                             "CX3CR1", "GZMB", "PRF1", "GNLY"], size = 20, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/30_CD8_Fig1.pdf", bbox_inches="tight",  dpi=400)
plt.show()

# "Response to type I IFN (GOBP)", "LAG3", "CXCR3", "ITGA4", "ITGB1"

In [ ]:
#Main figure material
fig_markers = sc.pl.umap(sdata["gex"], color=["Response to type I IFN (GOBP)", "BACH2", "LAG3", "MRPS6"], size = 20, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/30_CD8_Fig1_part2.pdf", bbox_inches="tight",  dpi=400)
plt.show()

# 

In [ ]:
## Supplemental figures
genes = [
    ## Naive
    "LEF1", "BACH2", "IL7R", "KLF2",
    #Memory
    "CD27", "CD28", "LTB", 
    #Effector fucntions
    "CCL5", "NKG7", "PRF1", "FASLG", "GZMH",
    "GZMA", "IFNG", "TNF", 
    "EOMES", "ZEB2", "TOX", "IKZF2", "PRDM1", "TBX21", 
    "PDCD1", "TIGIT", "HAVCR2", 
    ## NK like cells
    "NCR3", "KIR2DL1", "KIR2DL3", "KIR2DL4", "KIR3DL1", "KIR3DL2", "KIR3DL3", 
]

fig_markers = sc.pl.umap(sdata["gex"], color=genes, size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/30_CD8_supplememts1.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
## Supplemental figures
genes = [
    ## HLA genes
    "HLA-DRA", "HLA-DRB1", "CD74", 
    # Migration
    "CCR4", "CCR5", "CXCR4", "CXCR6", "CX3CR1", "S1PR1", "GPR183", 
    "ITGA4", "ITGB1", "ITGB7", "ITGAL",  "ITGB2", 
    # Activation
    "CD69", "BAX", "RELA", "NFKB2", "JUNB", "FOS", "NFATC3", "BATF", "BCL3", "TNFAIP3", "TGFB1", 
    # Prolifferation
    "MKI67",
    #Celldeath
    "MALAT1", 
    #Resident missing
    "XCL1", 
]


fig_markers = sc.pl.umap(sdata["gex"], color=genes, size = 20, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/30_CD8_supplememts2.pdf", bbox_inches="tight",  dpi=400)
plt.show()

### Report on doublets

In [ ]:
confusion_matrix = pd.crosstab(
    sdata["gex"].obs["L3_ann"],
    sdata["gex"].obs["chain_pairing"],
    rownames=['L3_ann'],
    colnames=['chain_pairing']
)


# Calculate row-wise percentages for the "two full chains" column
confusion_matrix["percent_two_full_chains"] = (
    confusion_matrix["two full chains"] / confusion_matrix.sum(axis=1)
) * 100

confusion_matrix


## Report final gene expression circle plot

In [ ]:
### Cluster order
obs_df = sdata["gex"].obs.copy()
obs_df[["leiden_1.8", "L3_ann"]]

## Order the dataframe according to: 
order_leiden = list(cluster_dict.keys())

## arrange the dataframe according to this order:
obs_df['leiden_1.8'] = pd.Categorical(obs_df['leiden_1.8'], categories=order_leiden, ordered=True)

# Sort the DataFrame according to this order
obs_df_sorted = obs_df.sort_values('leiden_1.8').reset_index(drop=True)

# Extract the ordered L3_ann list
order_l3 = obs_df_sorted["L3_ann"].drop_duplicates().tolist()

In [ ]:
## arrange the dataframe according to this order:
sdata["gex"].obs['leiden_1.8'] = pd.Categorical(sdata["gex"].obs['leiden_1.8'], categories=order_leiden, ordered=True)

marker_genes = {
    "CITEseq": ["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"],
    "General": ["CD3E", "TRDV2", "TRGV3"],
    "Naive": ["LEF1", "CCR7", "TCF7", "SELL", "STAT1"],
    "Memory": ["KLF2", "LTB", "IL7R"],
    "IFN-sig": ["IFI6", "ISG15", "IFI44L", "IFI44"],
    "Effector": [ "CCL5", "NKG7", "GZMH", "PRF1", "GZMB", "GNLY", "GZMA", "GZMK", "IFNG", "IL21"],
    "HLA": ["CD74", "HLA-DRA", "HLA-DRB1"],
    "Differentiation": [ "CX3CR1", "KLRG1", "IKZF2", "EOMES", "TOX", "TIGIT", "LAG3", "ZEB2", "KIR3DL1", "KIR3DL3", "TYROBP"],
    "Resident memory": ["ZNF683", "XCL1"],
    "Activation": ["CD44", "PDE4B", "CLDND1", "CD69", "NR4A1", "RELA", "NFKB2", "JUNB", "FOS", "TGFB1", "BAX"],
    "Homing": ["CCR4", "CCR5", "CXCR3", "CXCR4", "CXCR6", "S1PR1", "GPR183"],
    "Integrins": ["ITGA4", "ITGB1", "ITGB7", "ITGB2", "ITGAL"],
    "Cell cycle": ["MKI67", "CDKN1B"],
    "Other": ["RPL28", "RPL11", "RPL18", "MT-ATP8", "MT-ND5", "MALAT1"]
}

fig =  sc.pl.dotplot(sdata["gex"], marker_genes, groupby="leiden_1.8", standard_scale="var", dendrogram=False, cmap="RdBu_r", return_fig=True, show=False)


plt.tight_layout()
fig.savefig("../../05_plots/11_CD8annotation/39_L3_ann_marker_genes.pdf", bbox_inches="tight",  dpi=400)
plt.show()

## Report on major phenotypes

In [ ]:
### Report on the major phenotypes
sdata_report = sdata.copy()
mu.pp.filter_obs(sdata_report["gex"], "L3_ann_overview", lambda x: (x.isin(["TN", "TCM", "TEM", "TEMRA", "TE", "TCSF"]))) 
sdata_report.update()

### 
sdata_report["gex"].obs["L3_ann_overview"] = sdata_report["gex"].obs["L3_ann_overview"].cat.remove_unused_categories()

In [ ]:
sdata_report["gex"].obs["L3_ann_overview"].value_counts()

In [ ]:
### Specific to activated and ribolo
marker_genes = {
    "CITEseq" : ["prot_CD45RA_adt", "prot_CD62L_adt", "prot_CD95_adt"], 
    "TN" : [ "CCR7", "SELL",  "TCF7", "LEF1"],
    # "IFN" : ["ISG15", "MX1", "STAT1"],
    "TCM" : ["IL7R",  "LTB",  "S1PR1", "ITGA4", ], 
    "TEM" : [ "GZMK", "CXCR3", "CCL5", "NKG7", "GZMH"], 
    "TEMRA & TE": ["PRF1", "GZMB", "GNLY", "CX3CR1"],
    "TCSF": [ "TXNIP", "S100A4",  "GZMA"]
    
}

# ## Order the dataframe according to: 
order_overivew = ["TN", "TCM", "TEM", "TEMRA", "TE", "TCSF"]

# #Implement order
sdata_report["gex"].obs["L3_ann_overview"] = sdata_report["gex"].obs["L3_ann_overview"].cat.reorder_categories(
    order_overivew, ordered=True
)


dp = sc.pl.dotplot(
    sdata_report["gex"],
    marker_genes,
    groupby="L3_ann_overview",
    standard_scale="var",
    dendrogram=False,
    cmap="RdBu_r",
    swap_axes=False,
    show=False,
    return_fig=True,
)

dp.savefig(
    "../../05_plots/11_CD8annotation/27_CD8_general.pdf",
    bbox_inches="tight",
    dpi=400
)

## Export flow-like plots with L3_ann annotations

### Step 1: Flow-like plots for CITEseq data

In [ ]:
obs_df = sdata["gex"].obs.copy()

#Remove CSF clusters
obs_df = obs_df[~obs_df["leiden_1.8"].isin(["3", "5", "14", "16", "18"])]
obs_df = obs_df[~obs_df["Compartment"].isin(["CSF"])]
obs_df["leiden_1.8"] = obs_df["leiden_1.8"].cat.remove_unused_categories()

## Order the dataframe according to: 
order_leiden = ['0', '2', '1', '11',  '13', '7', '8', '4', '9', '10', '12', '6', '17', '19', '20', '22', '21', '15']

#Implement order
obs_df["leiden_1.8"] = obs_df["leiden_1.8"].cat.reorder_categories(
    order_leiden, ordered=True
)


In [ ]:
# -----------------------------------
# PARAMETERS (RAW ADT SPACE)
# -----------------------------------
x_gate = 14
y_gate = 13

xlim = (-5, 61)
ylim = (-5, 41)

# -----------------------------------
# LOOP OVER L3 CATEGORIES
# -----------------------------------
for l3 in obs_df["L3_ann"].dropna().unique():

    df_sub = obs_df[obs_df["L3_ann"] == l3]
    leiden_name = df_sub["leiden_1.8"].drop_duplicates().unique()[0]

    if len(df_sub) < 50:
        print(f"Skipping {l3} (n={len(df_sub)})")
        continue

    x = df_sub["prot_CD45RA_adt"]
    y = df_sub["prot_CD62L_adt"]

    # -----------------------------------
    # JOINT KDE PLOT (new figure per L3)
    # -----------------------------------
    g = sb.jointplot(
        data=df_sub,
        x="prot_CD45RA_adt",
        y="prot_CD62L_adt",
        kind="kde",
        fill=True,
        bw_adjust=0.50,
        levels=10,
        thresh=0.05,
        clip=((-5, 60), (-5, 40)),
        height=5
    )

    ax = g.ax_joint

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # -----------------------------------
    # GATING LINES
    # -----------------------------------
    ax.axvline(x_gate, color="black", linestyle="--")
    ax.axhline(y_gate, color="black", linestyle="--")

    sb.despine(ax=ax)

    # -----------------------------------
    # QUADRANT PERCENTAGES (PER L3)
    # -----------------------------------
    total = len(df_sub)

    q1 = (x >= x_gate) & (y >= y_gate)
    q2 = (x < x_gate) & (y >= y_gate)
    q3 = (x < x_gate) & (y < y_gate)
    q4 = (x >= x_gate) & (y < y_gate)

    pcts = [
        100 * q1.sum() / total,
        100 * q2.sum() / total,
        100 * q3.sum() / total,
        100 * q4.sum() / total,
    ]

    # -----------------------------------
    # QUADRANT ANNOTATION
    # -----------------------------------
    dx = 0.05 * (xlim[1] - xlim[0])
    dy = 0.10 * (ylim[1] - ylim[0])

    ax.text(x_gate + dx, y_gate + dy, f"{pcts[0]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate + dy, f"{pcts[1]:.1f}%", fontsize=14)
    ax.text(x_gate - 3*dx, y_gate - 2*dy, f"{pcts[2]:.1f}%", fontsize=14)
    ax.text(x_gate + dx, y_gate - 2*dy, f"{pcts[3]:.1f}%", fontsize=14)

    # -----------------------------------
    # TITLE + EXPORT
    # -----------------------------------
    ax.set_title(f"{l3} (n={total:,})")

    plt.tight_layout()
    plt.savefig(
        f"../../05_plots/11_CD8annotation/41_CD8_CD45RA_CD62L_{leiden_name}.pdf",
        dpi=400,
        bbox_inches="tight"
    )
    plt.show()


### Step 2: Correspondance
How well my GEX annotations recapitulate CITEseq annotations?

In [ ]:
obs_df # The DF from the virtual gate plots before. 

# Crosstab (rows = Quadrant, columns = L3 annotation)
ct = pd.crosstab(obs_df["L3_ann"], obs_df["Quadrant_final"])

# Sort rows by total number of cells
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=True).index]

# Plot horizontal stacked bar chart
ax = ct.plot(
    kind="barh",       # horizontal bars
    stacked=True,
    figsize=(9, 5),
    colormap="tab20"   # nice color palette
)

plt.ylabel("Leiden clusters")  # now vertical axis
plt.xlabel("Number of cells")          # horizontal axis
plt.title("")
plt.legend(title="CITEseq annotations", bbox_to_anchor=(1.02, 1), loc="upper left")

#Despine
sb.despine()
plt.grid(False)

plt.tight_layout()
plt.savefig(
    "../../05_plots/11_CD8annotation/42_CD8_CITEseq_reannotation_L3_ann.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# # Crosstab (rows = L3 annotation, columns = Quadrant)
# ct = pd.crosstab(df_sub["L3_ann_overview"], df_sub["Quadrant_final"])

# # Normalize rows to sum to 1 (fraction per row)
# ct_norm = ct.div(ct.sum(axis=1), axis=0)

# # Sort rows by total number of cells (optional, now using original counts)
# ct_norm = ct_norm.loc[ct.sum(axis=1).sort_values(ascending=True).index]

# # Plot horizontal stacked bar chart
# ax = ct_norm.plot(
#     kind="barh",       # horizontal bars
#     stacked=True,
#     figsize=(4, 5),
#     colormap="tab20"   # nice color palette
# )

# plt.ylabel("Gene expression annotations")          # now vertical axis
# plt.xlabel("Fraction of cells")      # horizontal axis
# plt.title("Normalized distribution across quadrants")
# plt.legend(title="CITEseq annotations", bbox_to_anchor=(1.02, 1), loc="upper left")
# plt.tight_layout()
# plt.savefig(
#     f"../../05_plots/08_AfterAntwerp/72_CD8_CITEseq_Gex_pct_correspondance.pdf",
#     dpi=400,
#     bbox_inches="tight"
# )
# plt.show()


In [ ]:
# Crosstab (rows = L3 annotation, columns = Quadrant)
ct = pd.crosstab(obs_df["Quadrant_final"], obs_df["leiden_1.8"])

quadrant_order = [
    "Tn: CD45RA+CD62L+CD95-",
    "Tscm: CD45RA+CD62L+CD95+",
    "Tcm: CD45RA-CD62L+",
    "Tem: CD45RA-CD62L-",
    "Temra: CD45RA+CD62L-"
]

# Reindex columns according to desired order
ct = ct.reindex(index=quadrant_order)

# Optionally normalize per row for proportions
ct_norm = ct.div(ct.sum(axis=0), axis=1) * 100  # percentages

# Plot heatmap
plt.figure(figsize=(14, 3))
sb.heatmap(
    ct_norm,
    annot=True,        # show numbers
    fmt=".1f",         # one decimal
    cmap="Reds",    # color map
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "% of cells per L3 annotation"}
)

plt.ylabel("CITEseq annotations")
plt.xlabel("Leiden 2")
plt.title("Phenotypic distribution across L3 clusters")
plt.tight_layout()
plt.savefig(
     "../../05_plots/11_CD8annotation/43_CD8_CITEseq_reannotation_correspondance_L3ann.pdf",
    dpi=400,
    bbox_inches="tight"
)
plt.show()

# Clean up

### Step 1: Transfer important information and make sure the clone_handle does not contain artefacts

In [ ]:
#Make into categorical columns
sdata["gex"].obs["donor"] = sdata["gex"].obs["donor"].astype("category")

#Transfer important information
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()

# Transfer remaining important information
sdata["airr"].obs["donor"] = sdata["gex"].obs["donor"].copy()
sdata["airr"].obs["pool"] = sdata["gex"].obs["pool"].copy()
sdata["airr"].obs["Compartment"] = sdata["gex"].obs["Compartment"].copy()
sdata["airr"].obs["clone_handle"] = sdata["gex"].obs["clone_handle"].copy()

# If clone_id is nan then clone_handle is also na
sdata["gex"].obs["clone_handle"] = np.where(sdata["gex"].obs["clone_id"].isna(), None, sdata["gex"].obs["clone_handle"])

sdata.update()

### Step 2: Make sure the GEX, AIRR and PROT modalities represent the same cells found in GEX

In [ ]:
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata["gex"].obs.index.tolist()

## define a cell index
sdata["airr"].obs["cell_index"] = sdata["airr"].obs.index
sdata["prot"].obs["cell_index"] = sdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata["airr"].obs and sdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata["prot"], 'filter_index', lambda x: x == "GEX_present")

### Step 3: Define clonotypes and type_x_clone_count (compartment-wise per clonotype)

In [ ]:
#Transfer labels from the airr to gex modality
# sdata["gex"].obs["antigen.species"] = sdata["airr"].obs["antigen.species"].copy()
# sdata["gex"].obs["antigen.epitope"] = sdata["airr"].obs["antigen.epitope"].copy()

#CDR3aa
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()
sdata["gex"].obs["IR_VJ_1_junction_aa"] = sdata["airr"].obs["IR_VJ_1_junction_aa"].copy()

#TRAV and TRBV
sdata["gex"].obs["IR_VJ_1_v_call"] = sdata["airr"].obs["IR_VJ_1_v_call"].copy()
sdata["gex"].obs["IR_VDJ_1_v_call"] = sdata["airr"].obs["IR_VDJ_1_v_call"].copy()

#D regions
sdata["gex"].obs["IR_VDJ_1_d_call"] = sdata["airr"].obs["IR_VDJ_1_d_call"].copy()

#J regions
sdata["gex"].obs["IR_VJ_1_j_call"] = sdata["airr"].obs["IR_VJ_1_j_call"].copy()
sdata["gex"].obs["IR_VDJ_1_j_call"] = sdata["airr"].obs["IR_VDJ_1_j_call"].copy()


In [ ]:
sdata_backup = sdata.copy()
airr_export = sdata["airr"].obs[["pool","donor","Compartment", "clone_id", "clone_handle",
                   'IR_VJ_1_junction_aa', 'IR_VDJ_1_junction_aa',
                   'IR_VJ_1_v_call', 'IR_VDJ_1_v_call',
                   'IR_VJ_1_j_call', 'IR_VDJ_1_j_call',
                   'IR_VDJ_1_d_call',
                   'IR_VJ_2_junction_aa', 'IR_VDJ_2_junction_aa',
                   'IR_VJ_2_v_call',  'IR_VDJ_2_v_call', 
                   'IR_VJ_2_j_call', 'IR_VDJ_2_j_call',
                   'IR_VDJ_2_d_call']]

# Drop rows where 'clone_handle' is NaN
airr_export = airr_export.dropna(subset=['Compartment']) #Subset tells pandas to dropna only for Comparment
airr_export = airr_export.dropna(subset=['clone_id']) #Subset tells pandas to dropna only for clone_id

## First generate a new clone ID label that is a concatenation of donor and clone_id
airr_export["clone_handle"] #Already generated

## Second generate a summary table with counts per clone_handle per compartment - clone_handles are used here to avoid counting public shared by donors together
counts_per_id = airr_export.groupby(["Compartment", 'clone_handle']).size().reset_index(name='type_x_clone_count')

# Merge the count information back into the original DataFrame
airr_export = pd.merge(airr_export, counts_per_id, on=["Compartment", 'clone_handle'], how='left')

#### Clean the data frame

In [ ]:
airr_export_backup = airr_export.copy()
airr_export = airr_export[["pool", "donor", "Compartment", "clone_id", "clone_handle", "type_x_clone_count", "IR_VJ_1_junction_aa", "IR_VDJ_1_junction_aa"]]

### Step 4: A venn diagram -- non-naive clonotypes

In [ ]:
## Non naive clone handles
NonNaive_set = set(sdata["gex"].obs[~sdata["gex"].obs["leiden_1.8"].isin(["0", "1", "2", "5"])]["clone_handle"])

In [ ]:
from matplotlib_venn import venn2

# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(5, 5))

# %%% Patient 1 %%%%
# Extract unique values from each column
set1 = set(airr_export[airr_export['pool'].isin(["Pt1_PBMC", "Pt4_PBMC", "Pt5_PBMC", "Pt8_PBMC", "Pt11_PBMC", "Pt12_PBMC", "Pt14_PBMC", "Pt17_PBMC", "Pt18_PBMC", "Pt19_PBMC", "Pt20_PBMC", "Pt21_PBMC"])]["clone_handle"])
set2 = set(airr_export[airr_export['pool'].isin(["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12", "Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt20+21"])]["clone_handle"])

#Non_naive set 1
set1_nonnaive = set1 & NonNaive_set

# Compute sizes for each section
only_set1 = len(set1_nonnaive - set2)
only_set2 = len(set2 - set1_nonnaive)
intersection = len(set1_nonnaive & set2)

total = only_set1 + only_set2 + intersection

# Compute percentages
pct1 = 100 * only_set1 / total
pct2 = 100 * only_set2 / total
pct_inter = 100 * intersection / total

# Create a Venn diagram
vennAll = venn2(subsets=(only_set1, only_set2, intersection), set_labels=('PB', 'CSF'))

# Modify labels to show percentages
if vennAll.get_label_by_id('10'):
    vennAll.get_label_by_id('10').set_text(f"{only_set1}\n({pct1:.1f}%)")
if vennAll.get_label_by_id('01'):
    vennAll.get_label_by_id('01').set_text(f"{only_set2}\n({pct2:.1f}%)")
if vennAll.get_label_by_id('11'):
    vennAll.get_label_by_id('11').set_text(f"{intersection}\n({pct_inter:.1f}%)")

# Set colors to a colorblind-friendly palette
vennAll.get_patch_by_id('10').set_color('#C06A6A')  # left = Muted red
vennAll.get_patch_by_id('01').set_color('#4B8AC1')  # right = Blue
vennAll.get_patch_by_id('11').set_color('#D89A52')  # overlap = orange

# Adjust transparency for all patches
for venn in [vennAll]:
    for patch in venn.patches:
        if patch:
            patch.set_alpha(1)

# Display the plot
plt.title("Clonotype overlap, CD8 T cells") 
plt.tight_layout()
plt.show()

# Save the figure 
fig.savefig("../../05_plots/13_final_plots/2_NonNaiveT_venn_overlap.pdf", bbox_inches='tight', format="pdf", dpi=400)

### Step 5: Binomial test
https://www.statology.org/binomial-test-python/

Example 1: We roll a 6-sided die 24 times and it lands on the number “3” exactly 6 times. Perform a binomial test to determine if the die is biased towards the number “3.”

**binom_test(x=6, n=24, p=1/6, alternative='greater')**

0.1995295129479586

Because this p-value (0.1995) is not less than 0.05, we fail to reject the null hypothesis. We do not have sufficient evidence to say the die is biased towards the number “3.”

#### Can we do the same for compartment resampling
From Kilian'sa advice: 
The clone size corresponds to the probability of sampling a given clone, fx. if the clone size is 30 out of 3000 cells, then the probability is 30/3000 or 1%
For any given clone this corresponds to a binomial distribution with the probability being equal to the clone fraction in PB, and
n being the number of T cells surveyed in the CSF repertoire. A binomal test can be made. 

So: binom_test(
    x=    "how many X clones we saw", 
    n=    "how many cells we sampled in CSF", 
    p=    "a baseline distribution based on resampling PB", 
    alternative='two-sided')

Under the assumption of random clone sharing between PB and CSF, we can assume that CSF can be "created" from the binomal distribution observed in the PB samples. 
The clone size in the PB sample corresponds to the probability of sampling a given clone. For any given clone this corresponds to a binomial distribution with the probability being equal
to the clone fraction in PB and n being the number of T cells surveyed in CSF. Using a binomial test, we can test if the observed size of a given clone in the CSF 
corresponds to the expected under the assumption of random clone sharing (observed CSF fraction should under the null hypothesis be similar to fraction in CSF * N cells in CSF) 
This method only simulates the probabilities of clone sharing. 

(Nikolaj) Why not just use Fischer's exact test?


In [ ]:
# Definition of cell total per sample
airr_export['cells_in_sample'] = airr_export.groupby(['pool'])['type_x_clone_count'].transform('size')

# %%%%%%%%%%%%
# Definition of clone_x_fra_PB (used as reference frequency later)
# %%%%%%%%%%%%

## For each clonotype calculate the fraction of cells out of all cells in PB. I.e. the fraction of the dataset made up by the given clone. 
PB_clones = airr_export[airr_export["Compartment"] == "PB"]
PB_clones = PB_clones[["donor", "clone_handle", "type_x_clone_count", "cells_in_sample"]]
PB_clones = PB_clones.drop_duplicates().reset_index()
PB_clones["clone_x_fra_PB"] = PB_clones["type_x_clone_count"] / PB_clones["cells_in_sample"]
PB_clones = PB_clones[["donor", "clone_handle", "clone_x_fra_PB"]]

## merge the summary information back
airr_export = pd.merge(airr_export, PB_clones, on=["donor", 'clone_handle'], how='left')

# Define minimum values as a lower obund too
dfc = PB_clones.groupby('donor')['clone_x_fra_PB']
Min_df = PB_clones.assign(min=dfc.transform(min))

Min_df = Min_df[["donor", "min"]].drop_duplicates()

## merge the min data information back
airr_export = pd.merge(airr_export, Min_df, on=["donor"], how='left')

## Now replace the NaN values in clone_x_fra_PB with the lower bound min value
airr_export['clone_x_fra_PB'] = airr_export['clone_x_fra_PB'].fillna(airr_export.pop('min'))

# %%%%%%%%%%%%
# Definition of clone_x_fra_CSF (used as reference frequency later)
# %%%%%%%%%%%%

## Get the fraction of cells for a given clone from the PB compartment per clone
CSF_clones = airr_export[airr_export["Compartment"] == "CSF"]
CSF_clones = CSF_clones[["donor", "clone_handle", "type_x_clone_count", "cells_in_sample"]]
CSF_clones = CSF_clones.drop_duplicates().reset_index()
CSF_clones["clone_x_fra_CSF"] = CSF_clones["type_x_clone_count"] / CSF_clones["cells_in_sample"]
CSF_clones = CSF_clones[["donor", "clone_handle", "clone_x_fra_CSF"]]

## merge the summary information back
airr_export = pd.merge(airr_export, CSF_clones, on=["donor", 'clone_handle'], how='left')

# Define minimum values as a lower obund too
dfc = CSF_clones.groupby('donor')['clone_x_fra_CSF']
Min_df = CSF_clones.assign(min=dfc.transform(min))

Min_df = Min_df[["donor", "min"]].drop_duplicates()

## merge the min data information back
airr_export = pd.merge(airr_export, Min_df, on=["donor"], how='left')

## Now replace the NaN values in clone_x_fra_PB with the lower bound min value
airr_export['clone_x_fra_CSF'] = airr_export['clone_x_fra_CSF'].fillna(airr_export.pop('min'))

In [ ]:
#How does the binom test work: Example: 
binomtest(6, 24, p=1/6)

In [ ]:
## For CSF cells compared to PB background distribution
airr_export["p_csf"] = airr_export.apply(lambda x: binomtest(x['type_x_clone_count'],   #k - number of successes (number of observed cells in the given compartment)
                                        x['cells_in_sample'],                       #n - number of trials (based on cells in the given compartment)
                                        x['clone_x_fra_PB'],                        #p - null expected likelihood of success (based on freq. in PB)
                                        alternative= "two-sided").pvalue, axis=1)

## For PB cells compared to CSF background distribution
airr_export["p_PB"] = airr_export.apply(lambda x: binomtest(x['type_x_clone_count'],   #k - number of successes (number of observed cells in the given compartment)
                                        x['cells_in_sample'],                       #n - number of trials (based on cells in the given compartment)
                                        x['clone_x_fra_CSF'],                        #p - null expected likelihood of success (based on freq. in PB)
                                        alternative= "two-sided").pvalue, axis=1)

## Merge the two p-values - take the smallest of the two first. 
airr_export["p"] = np.where(airr_export["p_csf"] < airr_export["p_PB"], airr_export["p_csf"], airr_export["p_PB"])

#### Bonferroni adj
The Bonferroni correction can also be applied as a p-value adjustment: Using that approach, instead of adjusting the alpha level, each p-value is multiplied by the number of tests (with adjusted p-values that exceed 1 then being reduced to 1), and the alpha level is left unchanged. The significance decisions using this approach will be the same as when using the alpha-level adjustment approach.

In [ ]:
## Correct the p-value by dividing with the number of test made for every donor
N_tests_df = airr_export[["donor", "Compartment", "clone_handle"]].drop_duplicates().reset_index()
N_tests_df["N_tests"] = N_tests_df.groupby(['donor', "Compartment"])['clone_handle'].transform('size')

# Merge the count information back into the original DataFrame
airr_export = pd.merge(airr_export, N_tests_df, on=["donor","Compartment",'clone_handle'], how='left')

In [ ]:
#Bonferroni p_adj calculation
airr_export["p_adj"] = airr_export["p"]*airr_export["N_tests"]  

#Mask p_adj-value to 1
airr_export['p_adj'] = airr_export['p_adj'].apply(lambda x: 1 if x > 1 else x)

#Generate a label if significant. 
airr_export["Sig_label"] = airr_export['p_adj'].apply(lambda x: "ns" if x >= 0.05 else "q < 0.05")

### Step 6: Biaxial plot w. cell counts

In [ ]:
### First you need the % values to later give color annotations correctly
# Add a column with the total number of observations for each combination
airr_export['cells_in_sample'] = airr_export.groupby(['pool'])['type_x_clone_count'].transform('size')

# Return unique values of two columns is in pandas defined as a "drop_duplicates()" problem
airr_export[["pool", "cells_in_sample"]].drop_duplicates() #Number of cells in sample seems realisitic - although 22 and 87 cells in CSF is going to be a problem...

#Define the percentage-wise observation from columns you have
airr_export['type_x_clone_pct'] = (airr_export['type_x_clone_count'] / airr_export['cells_in_sample']) * 100

# General columns to keep
cols = ["Compartment", "clone_handle", "type_x_clone_pct"] 

# %%% Patient samples %%%
PT_all = airr_export # All of it [airr_export["pool"].isin(["Pt1_PBMC", "Pt1", "Pt4_PBMC", "Pt4", "Pt5", "Pt5_PBMC", "Pt8", "Pt8_PBMC", "Pt11", "Pt11_PBMC", "Pt12", "Pt12_PBMC"])]
PT_all = PT_all[cols]
PT_all = PT_all.drop_duplicates(subset=cols)

#df.pivot(index='foo', columns='bar', values='baz')
PT_all = PT_all.pivot(index=["clone_handle"], columns=["Compartment"], values="type_x_clone_pct").fillna(0).reset_index()

#Rename to make merging easier later
Pct_df = PT_all.rename(columns={
    "CSF":"Pct_CSF", 
    "PB": "Pct_PB"
})

In [ ]:
# General columns to keep
cols = ["Compartment", "clone_handle", "type_x_clone_count", "Sig_label"]

# Assign colors based on categories
colors = {'ns': 'grey', 'q < 0.05': "red"}

# %%% Patient samples %%%
PT_all = airr_export
PT_all = PT_all[cols]
PT_all = PT_all.drop_duplicates(subset=cols)

# Pivot table
PT_all = PT_all.pivot(index=["clone_handle", "Sig_label"], columns=["Compartment"], values="type_x_clone_count").fillna(0).reset_index()

PT_all = PT_all.merge(Pct_df, how="left", on="clone_handle")

#Define color
PT_all["color"] = np.where((PT_all["Sig_label"] != "ns") & (PT_all["Pct_CSF"] > PT_all["Pct_PB"]), "CSF-enriched",
                          np.where((PT_all["Sig_label"] != "ns") & (PT_all["Pct_CSF"] < PT_all["Pct_PB"]), "PB-enriched", "ns"))

In [ ]:
# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(6.75, 4))

#color dictionary
color_map = {
    "PB-enriched": "#C06A6A",   # red
    "CSF-enriched":  "#4B8AC1",   # blue
    "ns":           "lightgrey"
}


# Plotting
x = PT_all["CSF"]
y = PT_all["PB"]

# Rasterize scatter plot & place grid behind data
axes.set_axisbelow(True)
scatter = axes.scatter(
    x=PT_all["CSF"],
    y=PT_all["PB"],
    alpha=1,
    s=50,
    c=PT_all["color"].map(color_map),
    edgecolors="white",
    linewidths=0.2,   # slim white border
    rasterized=True
)

# Labels and grid
axes.set_ylim(-55,1300)
axes.set_xlim(-5, 90)
axes.set_xlabel(r'CSF - No. of cells', fontsize=15)
axes.set_ylabel(r'PB - No. of cells', fontsize=15)
axes.set_title('Clonotype overlap, CD8 T cells, CSF vs PB T cells')
axes.grid(False)
sb.despine()

# Legend (explicitly tied to color_map)
legend_handles = [
    mpatches.Patch(color=color_map[label], label=label)
    for label in ["CSF-enriched", "PB-enriched", "ns"]
    if label in color_map
]

axes.legend(
    handles=legend_handles,
    title="Significance",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0.
)

# Layout and save
plt.tight_layout(rect=[0, 0, 0.85, 1])
fig.savefig("../../05_plots/13_final_plots/1_XY_expansion.pdf",
            bbox_inches='tight', format="pdf", dpi=400)
plt.show()

#### Save the statistical output

In [ ]:
#Is clone counts in airr_export the same as airr_backup - yes
len(airr_export[airr_export_backup["type_x_clone_count"] == airr_export["type_x_clone_count"]]) == len(airr_export)

### Step 7: Add back the clonal expansion information to the mudata object (sdata)

In [ ]:
#Generating a total count - adding back in sdata

# Create deduplicated DataFrame
airr_total_count = (
    airr_export[["clone_handle", "Compartment", "type_x_clone_count", "p_csf", "p_PB", "p_adj", "Sig_label"]]
    .drop_duplicates()
    .sort_values(by="clone_handle")
)

# Add total_clone_count column using transform to preserve row count
airr_total_count["total_clone_count"] = (
    airr_total_count
    .groupby("clone_handle")["type_x_clone_count"]
    .transform("sum")
)

# (Optional) Sort by total_clone_count
airr_total_count = airr_total_count.sort_values(by="total_clone_count", ascending=False)
airr_total_count

In [ ]:
airr_total_count[["clone_handle", "total_clone_count"]].sort_values("total_clone_count",ascending=False)

In [ ]:
## %%
# Add "type_x_clone_count" as well as "total_clone_count" back into the single-cell object
# %%

sdata["gex"].obs = sdata["gex"].obs.reset_index().merge(airr_total_count, on=["clone_handle", "Compartment"], how="left").set_index('index')  
  #Note: The reset_index to .set_index("index") works to save the current index of the left data frame, and then use that later in set_index() again. 
  #It's a common problem described here: https://stackoverflow.com/questions/11976503/how-to-keep-index-when-using-pandas-merge

sdata.update()

In [ ]:
sdata["gex"].obs["total_clone_count"] = sdata["gex"].obs["total_clone_count"].fillna(0)

## Final clone umap of clonal expansion

In [ ]:
## Note that total_clone_count is the more correct data frame
sdata["gex"].obs[sdata["gex"].obs["clone_size"]!= sdata["gex"].obs["total_clone_count"]][["clone_handle", "clone_size", "total_clone_count"]].drop_duplicates().head(10)

In [ ]:
## Create clonal bins
# Define binned categories
sdata["gex"].obs["clone_count_binned"] = pd.Categorical(np.where(sdata["gex"].obs["total_clone_count"] == 0, "NaN", 
                                                                 np.where(sdata["gex"].obs["total_clone_count"] == 1, "1", 
                                                                         np.where(sdata["gex"].obs["total_clone_count"] < 6, "2-5",  
                                                                                  np.where(sdata["gex"].obs["total_clone_count"] < 11, "6-10",
                                                                                           np.where(sdata["gex"].obs["total_clone_count"] < 21, "11-20", "> 20"))))))

#color dictionary
binned_map = {
    "NaN": "#FFFFFF",   # white
    "1":  "#F0F0F0",   # grey1
    "2-5":  "#D9D9D9",   # grey2
    "6-10":  "#BDBDBD",   # grey3
    "11-20":  "#737373",   # grey4
    "> 20":  "#252525",   # grey5
}

sc.settings.set_figure_params(figsize=(4.3, 4))
# Create the plot but do NOT show it
sc.pl.umap(
    sdata["gex"],
    color="clone_count_binned",
    palette=binned_map,
    size=60,
    add_outline=True,
    legend_fontweight="bold",
    legend_fontsize=6,
    frameon=False,
    show=False,
    legend_loc="on data",
    title=""
)

# Save exactly like your example
plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/4_Clonal_bin_umap.pdf",
    bbox_inches="tight",
    dpi=400
)
plt.show()

# Export L3_annotations

In [ ]:
sdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_CD8_annotated.h5mu")

In [ ]:
sdata["gex"].obs.columns

# Graveyard

In [ ]:
# --- Prepare obs_df ---
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["clone_size"] > 20]
obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 


# Leiden cluster
cls = "leiden_1.8"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

# --- Statistics: Mann–Whitney U per cluster ---
def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "RRMS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"] == "Control", "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan

# --- Plot ---
plt.figure(figsize=(10, 4))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    showfliers=False,
    linewidth=1
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=0.5,
    size=5,
    linewidth=0,
    ax=ax
)

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = pvals.get(cl, np.nan)
    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08
    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )
    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

plt.tight_layout()
plt.show()

pvals_df = (
    pd.DataFrame.from_dict(pvals, orient="index", columns=["p_value"])
    .reset_index()
    .rename(columns={"index": cls})
)
print(pvals_df)


### Alternative selection - Recluster on memory cells

In [ ]:
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8.1", lambda x: (~x.isin(["0", "1", "2", "4","6", "29"])))  
sdata.update()

In [ ]:
# n_pcs is set to 50 after advice on how to select the number of dimensions that goes into calculating neighbors on large datasets from Seurat: https://satijalab.org/seurat/archive/v3.0/mca
sc.pp.neighbors(sdata_temp["gex"], n_pcs=30, n_neighbors=15)  #Looking at n_pcs the variance ratio (explained variability) saturated close to 50
sc.tl.leiden(sdata_temp["gex"], key_added="leiden_1", resolution=1)
sc.tl.leiden(sdata_temp["gex"], key_added="leiden_1.8", resolution=1.5)
sc.tl.leiden(sdata_temp["gex"], key_added="leiden_1.8", resolution=2)

In [ ]:
sc.tl.umap(sdata_temp["gex"], spread=2)

In [ ]:
sc.pl.umap(sdata_temp["gex"], color=["leiden_1", "leiden_1.8","leiden_1.8"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, show=True, legend_loc = "on data")

In [ ]:
# --- Prepare obs_df ---
obs_df = sdata_temp["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["clone_size"] > 10]
obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 

# Leiden cluster
cls = "leiden_1.8"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

# --- Statistics: Mann–Whitney U per cluster ---
def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "RRMS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"] == "Control", "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan

# --- Plot ---
plt.figure(figsize=(10, 4))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    showfliers=False,
    linewidth=1
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=0.5,
    size=5,
    linewidth=0,
    ax=ax
)

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = pvals.get(cl, np.nan)
    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08
    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )
    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

plt.tight_layout()
plt.show()

pvals_df = (
    pd.DataFrame.from_dict(pvals, orient="index", columns=["p_value"])
    .reset_index()
    .rename(columns={"index": cls})
)
print(pvals_df)


In [ ]:
# --- Prepare obs_df ---
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["clone_size"] > 20]
obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 


# Leiden cluster
cls = "leiden_1.8"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

# --- Statistics: Mann–Whitney U per cluster ---
def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "RRMS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"] == "Control", "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan

# --- Plot ---
plt.figure(figsize=(10, 4))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    showfliers=False,
    linewidth=1
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=0.5,
    size=5,
    linewidth=0,
    ax=ax
)

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = pvals.get(cl, np.nan)
    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08
    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )
    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

plt.tight_layout()
plt.show()

pvals_df = (
    pd.DataFrame.from_dict(pvals, orient="index", columns=["p_value"])
    .reset_index()
    .rename(columns={"index": cls})
)
print(pvals_df)


# Scrublet check

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_1.8")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()



In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_1.8")["n_genes_by_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="n_genes_by_counts",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("n_genes_by_counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()



In [ ]:
len(sdata["gex"].obs[sdata["gex"].obs["leiden_1.8"] == "26"])  #59 cells. 

In [ ]:
### Exclude doublets when only CD8 T cells are studied
# mu.pp.filter_obs(sdata["gex"], "chain_pairing", lambda x: (x != "two full chains")) 
mu.pp.filter_obs(sdata["gex"], "leiden_1.8", lambda x: (x != "26")) 
#Remove unused cateorgies
sdata["gex"].obs["leiden_1.8"] = sdata["gex"].obs["leiden_1.8"].cat.remove_unused_categories()
sdata.update()

## Doublet scores

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_1.8")["scrublet_score"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="scrublet_score",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("scrublet_score")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()



In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_1.8")["n_genes_by_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="n_genes_by_counts",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("n_genes_by_counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()



In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("leiden_1.8")["total_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="total_counts",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("total_counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
# plt.savefig(
#     "..\\..\\05_plots\\08_AfterAntwerp\\88_Total_reads_L2_box.pdf",
#     bbox_inches="tight",
#     dpi=300
# )
plt.show()



In [ ]:
# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="IKZF2",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("IKZF2")


# CD45 and CD62L violin

In [ ]:
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: (x != "CSF"))
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: (~x.isin(["3", "4", "18", "19","20"])))
sdata_temp["gex"].obs["leiden_1.8"] = sdata_temp["gex"].obs["leiden_1.8"].cat.remove_unused_categories()

# Compute medians per group
medians = sdata_temp["gex"].obs.groupby("leiden_1.8")["prot_CD45RA_adt"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata_temp["gex"],
        keys="prot_CD45RA_adt",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("prot_CD45RA_adt")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
plt.savefig(
   "../../05_plots/11_CD8annotation/7_prot_CD45RA_adt_violin.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

# %%%%%%%%%%
# CD62L sorted by CD45RA
# %%%%%%%%%%

# Compute medians per group
medians = sdata_temp["gex"].obs.groupby("leiden_1.8")["prot_CD45RA_adt"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (10, 4)}):
    ax = sc.pl.violin(
        sdata_temp["gex"],
        keys="prot_CD62L_adt",
        groupby="leiden_1.8",
        order=sorted_groups,  # sorted by median
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("prot_CD62L")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
plt.savefig(
   "../../05_plots/11_CD8annotation/8_prot_CD62L_adt_violin.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


## Step 7: Specific DEGs

### Naive makers and non-naive markers (memory + effector) 

In [ ]:
#All naive and memory markers
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.8"].isin(["0", "1", "2", "5", ]), "Tn (0, 1, 2, 15)", "non-Tn"))
sc.tl.rank_genes_groups(sdata["gex"], groupby="deg_grp", groups=["Tn (0, 1, 2, 15)"], reference="non-Tn", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="deg_grp",  groups=["non-Tn"], reference="Tn (0, 1, 2, 15)", key_added="upY")


In [ ]:
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)

In [ ]:
#All naive and memory markers
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.8"].isin(["0", "1", "2"]), "Tn (0, 1, 2, 15)", sdata["gex"].obs["leiden_1.8"]))
# sc.tl.rank_genes_groups(sdata["gex"], groupby="deg_grp", groups=["5"], reference="Tn (0, 1, 2, 15)", key_added="upX")
sc.tl.rank_genes_groups(sdata["gex"], groupby="deg_grp", groups=["Tn (0, 1, 2, 15)"], reference="5", key_added="upY")

sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upY", n_genes=20)

In [ ]:
#All naive and memory markers
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["5"], reference="0", key_added="upX")

sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)

### T effector - DEGs

In [ ]:
#All naive and memory markers
sc.tl.rank_genes_groups(sdata["gex"], groupby="leiden_1.8", groups=["15"], reference="9", key_added="upX")

sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata["gex"], key="upX", n_genes=20)

In [ ]:
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upY", n_genes=20)
sc.settings.set_figure_params() #Back to default

In [ ]:
#All internal naive DEGs
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.8"].isin(["9", "10", "12", "13", "14"]), "TE", "non-TE"))
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "deg_grp", lambda x: (x == "TE"))
sdata_temp["gex"].obs["leiden_1.8"] = sdata_temp["gex"].obs["leiden_1.8"].cat.remove_unused_categories()
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="leiden_1.8", key_added="upX")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upX", n_genes=20, ncols=3)
sc.settings.set_figure_params() #Back to default

### T Activated - DEGs

In [ ]:
#All naive and memory markers
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.8"].isin(["17", "20", "21", "23", "25"]), "TA (17, 20, 21, 23, 25)", "non-TA, non-TN"))
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: (~x.isin(["0", "1", "2", "6", "15"])))
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp", groups=["TA (17, 20, 21, 23, 25)"], reference="non-TA, non-TN", key_added="upX")
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="deg_grp",  groups=["non-TA, non-TN"], reference="TA (17, 20, 21, 23, 25)", key_added="upY")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upX", n_genes=20)

In [ ]:
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upY", n_genes=20)
sc.settings.set_figure_params() #Back to default

In [ ]:
#All internal naive DEGs
sdata["gex"].obs["deg_grp"] = pd.Categorical(np.where(sdata["gex"].obs["leiden_1.8"].isin(["17", "20", "21", "23", "25"]), "TA", "non-TA"))
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "deg_grp", lambda x: (x == "TA"))
sdata_temp["gex"].obs["leiden_1.8"] = sdata_temp["gex"].obs["leiden_1.8"].cat.remove_unused_categories()
sc.tl.rank_genes_groups(sdata_temp["gex"], groupby="leiden_1.8", key_added="upX")
sc.settings.set_figure_params(figsize=(3, 2.5))
sc.pl.rank_genes_groups(sdata_temp["gex"], key="upX", n_genes=20, ncols=3)
sc.settings.set_figure_params() #Back to default

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["TCF7","CCR7", "SELL", "LEF1", "CD27", "CD28", "IL7R", "LTB", "KLF2", "BACH2", "prot_CD45RA_adt", "prot_CD62L_adt"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/30_CD8_NaiveAndCM.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["GZMB", "GZMA", "GZMH", "GZMK", "PRF1", "CCL5", "GNLY", "NKG7", "TNF", "IFNG", "FAS", "prot_CD95_adt"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/31_CD8_EffectorAssociated.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["TYROBP", "THEMIS","KLRB1", "KLRD1", "KLRC1","KLRC2", "KLRK1", "NCR3", "KIR3DL1", "KIR3DL2", "KIR2DL3", "TRDV1", "TRDV2", "TRGV3", "TRGV8", "TRBC1", "TRBC2", "TRAC"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/32_CD8_NK&GDAssociated.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
t_cell_tfs = [
    "TCF7",
    "LEF1",
    "FOXO1",
    "BACH2",
    "KLF2",
    "FOXP1",
    "TOX",
    "TBX21",
    "RUNX3",
    "BATF3",
    "ZEB2",
    "ID2",
    "STAT4",
    "PRDM1",
    "NR4A1",
    "NR4A2",
    "NR4A3",
    "MAF",
    "IKZF2",
    "BCL6",
    "AHR",
    "BCL11B",
    "TCF3",
    "RUNX1",
    "EOMES",
    "BCL3"
]


fig_markers = sc.pl.umap(sdata["gex"], color=t_cell_tfs, size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/33_CD8_TFs.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["CCR4", "CCR5", "CXCR3", "CXCR4","CXCR6", "CX3CR1", "ITGA4", "ITGB1", "ITGB2", "ITGAE", "ITGAL", "S1PR1", "ITGB7", "GPR171"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/34_CD8_homing_and_adhesions.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["PDCD1", "LAG3", "TIGIT", "HAVCR2", "LGALS3", "CTLA4", "CD244", "ENTPD1", "TNFRSF9", "TNFRSF18"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/35_CD8_exhaustion.pdf", bbox_inches="tight",  dpi=400)
plt.show()

In [ ]:
fig_markers = sc.pl.umap(sdata["gex"], color=["leiden_1.8", "chain_pairing"], size = 10, legend_fontweight= 'bold', frameon= False , legend_fontoutline=2, cmap="RdBu_r", return_fig=True)
fig_markers.savefig("../../05_plots/11_CD8annotation/36_CD8_misc.pdf", bbox_inches="tight",  dpi=400)
plt.show()